# 🌍🌙 AI Agent “Night Lights” — analysing a country's NTL with Groq, Gemini or Ollama

**STG17 Workshop** · *Emerging Issues, Emerging Practice* — Day 1 (*From RAG to Agent*) × Day 4 (*Africa after dark*)

This notebook builds, step by step, an **AI agent** that answers natural-language questions about the **night-time lights (NTL)** of a country identified by its **ISO3 code**. The agent relies on a **toolbox of 21 functions** (time series, subnational statistics, anomaly detection, unlit population, validation against World Bank data, event impact, quality control, maps and charts) powered by **Google Earth Engine**.

| Runs on | Brain (LLM) | Data |
|---|---|---|
| ✅ Google Colab · ✅ Kaggle · ✅ Local Jupyter / VS Code | Groq · Gemini · Ollama (through **a single** OpenAI-compatible interface) | VIIRS (NOAA/EOG) · Black Marble (NASA) · WorldPop · geoBoundaries · World Bank API |

**Notebook outline**

| # | Section | What you learn |
|---|---|---|
| 00 | How to use | Keys, accounts and prerequisites for each environment |
| 01 | Settings | Choosing the country, the LLM provider and the data source |
| 02–03 | Installation · visual identity | A portable notebook and a consistent visual style |
| 04 | The brain: LLM | One interface for three providers |
| 05 | NTL data | VIIRS products, artefacts, Earth Engine or demo mode |
| 06 | The toolbox | Designing tools an LLM can use without mistakes |
| 07 | Testing the tools by hand | Understanding what the agent “sees” |
| 08 | The agent | The *reason → act → observe* loop and answer formatting |
| 09–10 | Demonstrations · Chat | Guided questions, then free conversation |
| 11–12 | Benchmark · Evaluation | Comparing providers, measuring reliability |
| 13 | Publication | HTML report ready for GitHub Pages |
| 14–15 | Extension · Limitations | Adding your own tools, staying honest about what a proxy cannot do |

> 💡 **Run the cells in order** (`Runtime › Run all` also works). Only cells 01 (settings) and 09 (questions) need editing.

In [ ]:
#@title 🌙 Cover
from IPython.display import HTML, display
display(HTML("""
<div style="position:relative;overflow:hidden;border-radius:16px;padding:40px 44px 52px;
            background:linear-gradient(135deg,#00553A 0%,#00704A 45%,#00A86A 100%);
            font-family:Calibri,Carlito,'Segoe UI',Arial,sans-serif;color:#fff;">
  <svg style="position:absolute;right:-70px;top:-80px;opacity:.16" width="460" height="460" viewBox="0 0 460 460">
    <g fill="none" stroke="#fff" stroke-width="2">
      <circle cx="230" cy="230" r="220"/><circle cx="230" cy="230" r="165"/>
      <circle cx="230" cy="230" r="110"/><circle cx="230" cy="230" r="55"/></g></svg>
  <svg style="position:absolute;right:40px;bottom:30px" width="330" height="150" viewBox="0 0 330 150">
    <g fill="#F5C242">
      <circle cx="40" cy="110" r="3" opacity=".9"/><circle cx="70" cy="95" r="2" opacity=".6"/>
      <circle cx="95" cy="120" r="5" opacity=".95"/><circle cx="130" cy="70" r="2.5" opacity=".7"/>
      <circle cx="160" cy="100" r="7" opacity="1"/><circle cx="170" cy="92" r="3" opacity=".8"/>
      <circle cx="205" cy="60" r="2" opacity=".5"/><circle cx="235" cy="115" r="4" opacity=".9"/>
      <circle cx="260" cy="80" r="2" opacity=".6"/><circle cx="290" cy="105" r="3" opacity=".8"/>
      <circle cx="120" cy="40" r="1.5" opacity=".5"/><circle cx="310" cy="50" r="1.5" opacity=".5"/></g></svg>
  <div style="letter-spacing:3px;font-size:12px;font-weight:700;color:#F5C242;text-transform:uppercase">
    STG17 · Technical Workshop · Africa after dark</div>
  <div style="font-size:42px;font-weight:700;margin:12px 0 6px;line-height:1.08">AI Agent “Night Lights”</div>
  <div style="font-size:18px;font-style:italic;color:#E6F6EE;max-width:720px">
    An augmented geospatial analyst: from a country's ISO code to subnational indicators,
    maps and a methodological note — in natural language.</div>
  <div style="margin-top:22px;display:flex;flex-wrap:wrap;gap:8px">
    <span style="background:rgba(255,255,255,.14);border:1px solid rgba(255,255,255,.35);border-radius:999px;padding:4px 12px;font-size:13px">🧠 Groq · Gemini · Ollama</span>
    <span style="background:rgba(255,255,255,.14);border:1px solid rgba(255,255,255,.35);border-radius:999px;padding:4px 12px;font-size:13px">🛰️ Google Earth Engine · VIIRS · Black Marble</span>
    <span style="background:rgba(255,255,255,.14);border:1px solid rgba(255,255,255,.35);border-radius:999px;padding:4px 12px;font-size:13px">🧰 21 tools</span>
    <span style="background:rgba(255,255,255,.14);border:1px solid rgba(255,255,255,.35);border-radius:999px;padding:4px 12px;font-size:13px">💻 Colab · Kaggle · Local</span>
  </div>
  <div style="position:absolute;left:0;right:0;bottom:0;height:8px;background:#F5C242"></div>
</div>
<div style="font-family:Calibri,Carlito,Arial,sans-serif;font-size:11px;color:#8A9590;margin-top:4px">
  Visual style inspired by the AfDB (unofficial) · Teaching notebook — STG17 workshop</div>
"""))

## 00 · How to use

### What you need

| Item | Required? | Where to get it |
|---|---|---|
| **An LLM key** — `GROQ_API_KEY` *or* `GEMINI_API_KEY` | Yes, except for Ollama | Groq: <https://console.groq.com/keys> · Gemini: <https://aistudio.google.com/apikey> (free tiers available) |
| **A Google Cloud project registered for Earth Engine** | Recommended | <https://code.earthengine.google.com/register> (free for non-commercial use), then enter the project ID in `GEE_PROJECT` |
| Ollama | Only if `LLM_PROVIDER="ollama"` | Local: <https://ollama.com/download> · Colab/Kaggle: installed automatically by section 04 (GPU recommended) |

> 🧪 **No Earth Engine account yet?** Leave `DATA_BACKEND="auto"`: without a GEE project, the notebook switches to **demo mode** with **synthetic** data (clearly flagged everywhere) so you can learn how the agent works. Demo-mode figures must **never** be quoted.

### Where to store keys (never in plain text in the notebook)

| Environment | Method |
|---|---|
| **Colab** | 🔑 *Secrets* icon (left sidebar) → add `GROQ_API_KEY` / `GEMINI_API_KEY` / `GEE_PROJECT` and enable “Notebook access”. Earth Engine: a Google authorisation window opens on first run. |
| **Kaggle** | *Add-ons › Secrets* → same names. Turn on **Internet** in the *Settings* panel. For Earth Engine, preferably add a `GEE_SERVICE_ACCOUNT_JSON` secret (JSON key of a service account registered for Earth Engine); otherwise you will be offered link/code authentication. |
| **Local** | Environment variables or a `.env` file next to the notebook (`GROQ_API_KEY=...`). Earth Engine: run `earthengine authenticate` once in a terminal. |

If a key cannot be found, the notebook asks for it in a masked field (`getpass`).

## 01 · Session settings

Choose **the country** (ISO 3166-1 alpha-3 code: `RWA`, `SEN`, `CIV`, `TUN`, `KEN`, `NGA`, `MOZ`, `CMR`, `SOM`…), **the provider** and **the data source**.
In Colab, these fields appear as a form.

| Setting | Purpose |
|---|---|
| `LLM_MODEL` | Empty = the provider's default model (see section 04). The model must support **tool calling** (*function calling*). |
| `TOOLSET` | `full` = 21 tools; `core` = 9 essential tools, recommended for **small local models** (Ollama 7–8B), which get lost when the catalogue is too long. |
| `LIT_THRESHOLD` | Threshold (nW/cm²/sr) above which a pixel is considered “lit”. 0.5 is a common value; test its effect. |

In [ ]:
#@title ⚙️ Session settings
COUNTRY_ISO3    = "RWA"        #@param {type:"string"}
LLM_PROVIDER    = "groq"       #@param ["groq", "gemini", "ollama"]
LLM_MODEL       = ""           #@param {type:"string"}
DATA_BACKEND    = "auto"       #@param ["auto", "gee", "demo"]
GEE_PROJECT     = ""           #@param {type:"string"}
ANSWER_LANGUAGE = "English"    #@param ["English", "français", "português"]
TOOLSET         = "full"       #@param ["full", "core"]
LIT_THRESHOLD   = 0.5          #@param {type:"number"}
MAX_AGENT_STEPS = 12           #@param {type:"integer"}

TEMPERATURE = 0.1                     # low = more stable, reproducible answers
OUTPUT_DIR  = "ntl_agent_outputs"     # figures, data, cache and report
COUNTRY_ISO3 = COUNTRY_ISO3.strip().upper()
print(f"Country: {COUNTRY_ISO3} · LLM: {LLM_PROVIDER} {LLM_MODEL or '(default model)'} · data: {DATA_BACKEND}")

## 02 · Installation and environment detection

The next cell automatically detects **Colab**, **Kaggle** or **local**, installs only the missing packages, then prepares the output folders. It is idempotent: you can safely re-run it.

In [ ]:
import os, sys, subprocess, importlib, importlib.util, platform, time, json, re, math, inspect
import hashlib, zlib, pickle, io, base64, difflib, textwrap, warnings, datetime as dt
from pathlib import Path
from typing import Optional, Literal, get_origin, get_args, Union

IN_COLAB  = "google.colab" in sys.modules
IN_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or Path("/kaggle/working").exists()
ENV_NAME  = "Google Colab" if IN_COLAB else ("Kaggle" if IN_KAGGLE else "Local")

PACKAGES = {  # imported module -> pip specification
    "openai": "openai>=1.40", "plotly": "plotly>=5.20", "pycountry": "pycountry",
    "markdown": "markdown", "ee": "earthengine-api", "shapely": "shapely>=2.0",
    "scipy": "scipy", "ipywidgets": "ipywidgets", "requests": "requests",
    "pandas": "pandas", "numpy": "numpy", "openpyxl": "openpyxl", "matplotlib": "matplotlib",
}
missing = [spec for mod, spec in PACKAGES.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("📦 Installing:", ", ".join(missing))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
    importlib.invalidate_caches()

import numpy as np
import pandas as pd
import requests
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
import markdown as md_lib
from IPython.display import HTML, Markdown, Image as IPImage, display

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
pd.set_option("display.max_columns", 30)

if IN_COLAB:
    pio.renderers.default = "colab"
elif IN_KAGGLE:
    pio.renderers.default = "notebook"

OUT = Path(OUTPUT_DIR)
for sub in ("figures", "data", "cache"):
    (OUT / sub).mkdir(parents=True, exist_ok=True)

print(f"✅ Environment: {ENV_NAME} · Python {platform.python_version()} · outputs in ./{OUT}")

## 03 · Visual identity and display components

A useful agent does not just compute correctly: it **presents** its results legibly. Here we define, once and for all:

* the **palette** (institutional green, gold as accent, brick red reserved for declines and risks) and a matching **Plotly theme**;
* small **HTML components** (cards, key indicators, alerts) reused by the agent.

> Each HTML output embeds its own style: in Colab, every cell is rendered in an isolated frame, so a global stylesheet would not be enough.

In [ ]:
C = dict(green="#00A86A", deep="#00704A", forest="#00553A", gold="#F5C242", ochre="#D49A00",
         teal="#0E7C86", terra="#C4621D", brick="#B83B2E", ink="#231F20", slate="#5E6964",
         mist="#F4F7F5", mint="#E8F5EF", sage="#D5DED9", grey="#A9B5B0")
RAMP   = ["#9ED9C0", "#7BCBA9", "#57BD92", "#33AF7C", "#10A06A", "#008A5B", "#00664A"]
SERIES = [C["green"], C["deep"], C["teal"], C["ochre"], C["terra"], C["brick"], "#7BCBA9", C["grey"]]
NIGHT  = ["#000814", "#061a33", "#0b2e4f", "#1d4e6b", "#b88a1c", "#F5C242", "#FFF3B0", "#FFFFFF"]
FONT   = "Calibri, Carlito, 'Segoe UI', Arial, sans-serif"

pio.templates["stg17"] = go.layout.Template(layout=dict(
    font=dict(family=FONT, size=13, color=C["ink"]),
    colorway=SERIES, paper_bgcolor="white", plot_bgcolor="white",
    title=dict(font=dict(size=18, color=C["ink"]), x=0.01, xanchor="left"),
    xaxis=dict(showgrid=False, linecolor=C["sage"], ticks="outside", tickcolor=C["sage"],
               tickfont=dict(color=C["slate"]), title=dict(font=dict(color=C["slate"]))),
    yaxis=dict(gridcolor="#E1E7E4", zeroline=False, linecolor=C["sage"],
               tickfont=dict(color=C["slate"]), title=dict(font=dict(color=C["slate"]))),
    legend=dict(bgcolor="rgba(0,0,0,0)", font=dict(color=C["slate"])),
    hoverlabel=dict(font=dict(family=FONT)),
    margin=dict(l=60, r=30, t=70, b=80),
))
pio.templates.default = "plotly_white+stg17"

BASE_CSS = """<style>
.ntl{font-family:Calibri,Carlito,'Segoe UI',Arial,sans-serif;color:#231F20;line-height:1.5;font-size:14.5px}
.ntl-card{background:#fff;border:1px solid #D5DED9;border-radius:12px;padding:18px 22px 16px;margin:10px 0;
  position:relative;overflow:hidden;box-shadow:0 1px 3px rgba(0,0,0,.05)}
.ntl-card:before{content:"";position:absolute;left:0;top:0;right:0;height:4px;
  background:linear-gradient(90deg,#F5C242 0 96px,#00A86A 96px)}
.ntl-card.hl{background:#E8F5EF;border:1.5px solid #00A86A}
.ntl-kicker{font-size:11px;font-weight:700;letter-spacing:3px;text-transform:uppercase;color:#00704A;margin-bottom:4px}
.ntl-title{font-size:20px;font-weight:700;color:#231F20;margin:0 0 6px}
.ntl-muted{color:#5E6964;font-size:12.5px}
.ntl h1,.ntl h2,.ntl h3,.ntl h4{color:#231F20;margin:14px 0 6px;line-height:1.25}
.ntl h3{font-size:17px;border-left:4px solid #00A86A;padding-left:9px}
.ntl table{border-collapse:collapse;margin:8px 0 10px;font-size:13.5px}
.ntl th{background:#E8F5EF;color:#00553A;text-align:left;padding:6px 10px;border-bottom:2px solid #00A86A}
.ntl td{padding:5px 10px;border-bottom:1px solid #D5DED9;vertical-align:top;text-align:left}
.ntl tr:nth-child(even) td{background:#FAFCFB}
.ntl code{background:#F4F7F5;padding:1px 5px;border-radius:4px;font-family:'Courier New',monospace;font-size:12.5px}
.ntl pre{background:#F4F7F5;border:1px solid #D5DED9;border-radius:8px;padding:10px;overflow:auto;font-size:12px}
.ntl-kpis{display:flex;flex-wrap:wrap;gap:10px;margin:10px 0 6px}
.ntl-kpi{flex:1 1 160px;background:#F4F7F5;border:1px solid #D5DED9;border-radius:10px;padding:10px 14px}
.ntl-kpi .v{font-size:26px;font-weight:700;color:#00704A;line-height:1.1}
.ntl-kpi .u{font-size:13px;color:#5E6964;font-weight:400}
.ntl-kpi .l{font-size:12px;color:#5E6964;margin-top:2px}
.ntl-kpi .d{font-size:12px;font-weight:700}
.ntl-up{color:#00A86A}.ntl-down{color:#B83B2E}
.ntl-pill{display:inline-block;padding:2px 10px;border-radius:999px;font-size:12px;border:1px solid #00A86A;
  background:#E8F5EF;color:#00553A;margin:2px 4px 2px 0}
.ntl-pill.gold{border-color:#D49A00;background:rgba(245,194,66,.15);color:#6b4d00}
.ntl-pill.red{border-color:#B83B2E;background:rgba(184,59,46,.08);color:#B83B2E}
.ntl-warn{background:rgba(212,154,0,.12);border:1px solid #D49A00;border-radius:10px;padding:10px 14px;margin:8px 0;color:#4d3800}
.ntl-alert{background:rgba(184,59,46,.08);border:1px solid #B83B2E;border-radius:10px;padding:10px 14px;margin:8px 0;color:#7a2219}
.ntl-ok{background:#E8F5EF;border:1px solid #00A86A;border-radius:10px;padding:10px 14px;margin:8px 0;color:#00553A}
.ntl details{margin-top:8px}.ntl summary{cursor:pointer;color:#00704A;font-weight:700}
.ntl-grid{display:grid;grid-template-columns:repeat(auto-fill,minmax(250px,1fr));gap:10px}
.ntl-tool{background:#F4F7F5;border:1px solid #D5DED9;border-radius:10px;padding:10px 12px}
.ntl-tool b{font-family:'Courier New',monospace;font-size:13px;color:#00553A}
</style>"""


def fmt(x, digits=2):
    """Readable format: 1234567 → '1.23 M'; 0.1234 → '0.12'."""
    if x is None or (isinstance(x, float) and (math.isnan(x) or math.isinf(x))):
        return "—"
    if isinstance(x, (int, np.integer)) and abs(x) < 1e6:
        return f"{x:,}".replace(",", " ")
    x = float(x)
    if abs(x) >= 1e9: return f"{x/1e9:.2f} bn"
    if abs(x) >= 1e6: return f"{x/1e6:.2f} M"
    if abs(x) >= 1e3: return f"{x:,.0f}".replace(",", " ")
    return f"{x:.{digits}f}"


class UI:
    """Small reusable HTML components (all self-styled)."""
    @staticmethod
    def show(body):
        display(HTML(BASE_CSS + f'<div class="ntl">{body}</div>'))

    @staticmethod
    def md(text):
        """Markdown → HTML, tolerating common LLM deviations (missing blank lines around
        tables, headings and lists)."""
        out, prev = [], ""
        for line in (text or "").splitlines():
            s = line.strip()
            is_table, was_table = s.startswith("|"), prev.startswith("|")
            bullet, num = re.match(r"^[-*] ", s), re.match(r"^\d+\. ", s)
            is_block = (s.startswith("#") or (bullet and not re.match(r"^[-*] ", prev))
                        or (num and not re.match(r"^\d+\. ", prev)))
            if prev and ((is_table and not was_table) or (was_table and not is_table and s) or is_block):
                out.append("")
            out.append(line)
            prev = s
        return md_lib.markdown("\n".join(out), extensions=["tables", "fenced_code", "sane_lists"])

    @staticmethod
    def card(title="", body="", kicker=None, highlight=False, raw=False):
        k = f'<div class="ntl-kicker">{kicker}</div>' if kicker else ""
        t = f'<div class="ntl-title">{title}</div>' if title else ""
        html = f'<div class="ntl-card{" hl" if highlight else ""}">{k}{t}{body}</div>'
        return html if raw else UI.show(html)

    @staticmethod
    def kpis(items):
        """items: list of dict(label, value, unit, delta)."""
        cells = []
        for it in items:
            d = it.get("delta")
            dh = ""
            if d is not None and not (isinstance(d, float) and math.isnan(d)):
                cls = "ntl-up" if d >= 0 else "ntl-down"
                dh = f'<div class="d {cls}">{"▲" if d >= 0 else "▼"} {abs(d):.1f} %</div>'
            cells.append(f'<div class="ntl-kpi"><div class="v">{it["value"]} <span class="u">{it.get("unit","")}</span></div>'
                         f'<div class="l">{it["label"]}</div>{dh}</div>')
        return f'<div class="ntl-kpis">{"".join(cells)}</div>'

    @staticmethod
    def pills(items, cls=""):
        return "".join(f'<span class="ntl-pill {cls}">{i}</span>' for i in items)

    @staticmethod
    def ok(msg):    UI.show(f'<div class="ntl-ok">✅ {msg}</div>')
    @staticmethod
    def warn(msg):  UI.show(f'<div class="ntl-warn">⚠️ {msg}</div>')
    @staticmethod
    def alert(msg): UI.show(f'<div class="ntl-alert">⛔ {msg}</div>')

ui = UI()
ui.card("Visual style loaded",
        ui.kpis([dict(label="Primary colour", value="#00A86A"), dict(label="Accent", value="#F5C242"),
                 dict(label="Plotly theme", value="stg17")]),
        kicker="03 · Visual identity")

### Overview: what we are going to build

An **agent** is an LLM placed in a loop: it reads the question, **decides** which tool to call, **observes** the result, and repeats until it can answer. The LLM computes nothing itself: **every figure comes from the tools**, which query Earth Engine. This principle is the best protection against hallucinations.

In [ ]:
def show_architecture():
    box = ("background:{bg};border:1.5px solid {bd};border-radius:12px;padding:12px 14px;min-width:150px;"
           "flex:1;text-align:center")
    arrow = '<div style="align-self:center;font-size:26px;color:#5E6964;padding:0 4px">⇄</div>'
    def b(title, sub, bg="#fff", bd="#00A86A", icon=""):
        return (f'<div style="{box.format(bg=bg, bd=bd)}"><div style="font-size:26px">{icon}</div>'
                f'<div style="font-weight:700;margin:2px 0">{title}</div><div class="ntl-muted">{sub}</div></div>')
    flow = ('<div style="display:flex;flex-wrap:wrap;gap:6px;align-items:stretch">'
            + b("Question", "in natural language<br>+ ISO3 code", icon="💬", bd="#D5DED9")
            + arrow + b("LLM (the brain)", "Groq · Gemini · Ollama<br>plans, chooses the tools", "#E8F5EF", icon="🧠")
            + arrow + b("Toolbox", "21 Python functions<br>auto-generated JSON schemas", icon="🧰", bd="#0E7C86")
            + arrow + b("Data", "Earth Engine: VIIRS, Black Marble,<br>WorldPop, geoBoundaries · World Bank", icon="🛰️", bd="#D49A00")
            + '</div>'
            '<div style="text-align:center;font-size:24px;color:#5E6964;margin:4px 0">⇩</div>'
            '<div style="display:flex;justify-content:center">'
            + b("Structured answer", "In brief · Results · Interpretation · <b>Limitations</b> · Method<br>"
                "+ key indicators, interactive charts, maps, call trace", "#fff", "#00704A", "📑")
            + '</div>')
    ui.card("Agent architecture", flow, kicker="Overview")

show_architecture()

## 04 · The brain: connecting to the LLM

### One interface for three providers

Groq, Gemini and Ollama all expose an **OpenAI-compatible API** (`/chat/completions` with *tool calling*). We therefore use **a single client** (`openai`), and only the server address changes. Switching provider = changing one setting: this is exactly the procurement situation you will compare during the workshop (Day 2, *Choosing your engine*).

| Provider | Where does the model run? | Strengths | Points of attention | Preferred model |
|---|---|---|---|---|
| **Groq** | Cloud (LPU chips) | Very fast, free tier | Rate limits (429 errors handled automatically); data sent to a third party | `openai/gpt-oss-120b` |
| **Gemini** | Google Cloud | Long context, good reasoning, free tier | Data sent to a third party; per-minute quotas | `gemini-2.5-flash` |
| **Ollama** | **Your machine** | **Sovereignty**: nothing leaves; free | Slow without a GPU; small models less reliable → use `TOOLSET="core"` | `qwen2.5:7b` |

> Model catalogues change quickly and models are regularly retired. The client **automatically checks** that the model exists for your key; if not, it falls back to the first available model from a preference list (all capable of calling tools) and tells you so. To force a model, set `LLM_MODEL` (section 01); `llm.list_models()` shows the exact list.

> 🔐 **Confidentiality**: the agent only sends **aggregates** (statistics by region) to the LLM, never microdata. For confidential data, prefer Ollama (local execution).

In [ ]:
import openai
from openai import OpenAI

# Catalogues change quickly (models are regularly retired): we provide an ORDERED list of preferred
# models that can call tools. At start-up, the client queries the list of models actually available
# for YOUR key and picks the first one present — no more "model_not_found" errors.
PROVIDERS = {
    "groq":   dict(base_url="https://api.groq.com/openai/v1", key_env="GROQ_API_KEY",
                   preferred=["openai/gpt-oss-120b", "moonshotai/kimi-k2-instruct-0905", "moonshotai/kimi-k2-instruct",
                              "qwen/qwen3-32b", "meta-llama/llama-4-maverick-17b-128e-instruct",
                              "meta-llama/llama-4-scout-17b-16e-instruct", "llama-3.3-70b-versatile",
                              "openai/gpt-oss-20b", "llama-3.1-8b-instant"]),
    "gemini": dict(base_url="https://generativelanguage.googleapis.com/v1beta/openai/", key_env="GEMINI_API_KEY",
                   preferred=["gemini-2.5-flash", "gemini-flash-latest", "gemini-2.5-pro", "gemini-2.5-flash-lite",
                              "gemini-2.0-flash"]),
    "ollama": dict(base_url=os.environ.get("OLLAMA_HOST", "http://localhost:11434").rstrip("/") + "/v1", key_env=None,
                   preferred=["qwen2.5:7b", "qwen3:8b", "llama3.1:8b", "mistral-nemo", "qwen2.5:14b"]),
}
for _cfg in PROVIDERS.values():
    _cfg["default_model"] = _cfg["preferred"][0]
    _cfg["alternatives"] = _cfg["preferred"][1:4]

# Models to exclude from automatic selection (audio, safety, embeddings…)
_NOT_CHAT = ("whisper", "tts", "guard", "embed", "playai", "orpheus", "compound", "aqa", "imagen", "veo",
             "image", "audio", "live", "safeguard")


def get_secret(name, prompt=True):
    """Looks up a secret: environment variable → Colab Secrets → Kaggle Secrets → .env file → masked input."""
    val = os.environ.get(name)
    if val:
        return val
    if IN_COLAB:
        try:
            from google.colab import userdata
            val = userdata.get(name)
        except Exception:
            val = None
    elif IN_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            val = UserSecretsClient().get_secret(name)
        except Exception:
            val = None
    if not val and Path(".env").exists():
        for line in Path(".env").read_text().splitlines():
            if line.strip().startswith(f"{name}="):
                val = line.split("=", 1)[1].strip().strip('"').strip("'")
    if not val and prompt:
        from getpass import getpass
        val = getpass(f"🔑 {name} (masked input): ").strip()
    if val:
        os.environ[name] = val
    return val or None


class LLMClient:
    """Single client (OpenAI-compatible API) for Groq, Gemini and Ollama, with automatic retry on 429."""

    def __init__(self, provider, model=None, prompt_for_key=True):
        if provider not in PROVIDERS:
            raise ValueError(f"Unknown provider: {provider}. Options: {list(PROVIDERS)}")
        cfg = PROVIDERS[provider]
        key = "ollama"
        if cfg["key_env"]:
            key = get_secret(cfg["key_env"], prompt=prompt_for_key)
            if not key:
                raise RuntimeError(f"Key {cfg['key_env']} not found.")
        self.provider = provider
        self.client = OpenAI(base_url=cfg["base_url"], api_key=key, timeout=180, max_retries=2)
        self.model = self._resolve_model(model or cfg["default_model"], cfg["preferred"])

    def _resolve_model(self, wanted, preferred):
        """Checks that the model exists for this key; otherwise automatically picks the best available one."""
        try:
            available = self.list_models()
        except Exception:
            return wanted                      # list unavailable: try the requested model
        if not available or wanted in available:
            return wanted
        choice = next((m for m in preferred if m in available), None)
        if choice is None:
            chat = [m for m in available if not any(k in m.lower() for k in _NOT_CHAT)]
            choice = (chat or available)[0]
        print(f"ℹ️ Model “{wanted}” unavailable at {self.provider} → automatically switching to “{choice}”.\n"
              f"   Available models: {', '.join(available[:15])}{' …' if len(available) > 15 else ''}")
        return choice

    def __repr__(self):
        return f"LLMClient({self.provider}:{self.model})"

    def chat(self, messages, tools=None, temperature=TEMPERATURE):
        kwargs = dict(model=self.model, messages=messages, temperature=temperature)
        if tools:
            kwargs.update(tools=tools, tool_choice="auto")
        for attempt in range(5):
            try:
                return self.client.chat.completions.create(**kwargs)
            except openai.RateLimitError:
                wait = min(60, 6 * 2 ** attempt)
                print(f"⏳ Rate limit reached at {self.provider} — retrying in {wait} s…")
                time.sleep(wait)
            except openai.APIConnectionError as e:
                if self.provider == "ollama":
                    raise RuntimeError("Ollama unreachable: start `ollama serve` (section 04) "
                                       "and check OLLAMA_HOST.") from e
                if attempt >= 1:
                    raise
                time.sleep(3)
        raise RuntimeError("Failed after several attempts (rate limit).")

    def list_models(self):
        """Models actually accessible with your key (Gemini's “models/” prefix removed)."""
        return sorted(m.id.removeprefix("models/") for m in self.client.models.list())

    def ping(self):
        t0 = time.time()
        r = self.chat([{"role": "user", "content": "Reply with only the word: OK"}], temperature=0)
        return (r.choices[0].message.content or "").strip(), time.time() - t0

### (Optional) Ollama on Colab / Kaggle

If you chose `ollama`, the next cell installs and starts the Ollama server **inside the Colab/Kaggle virtual machine**, then downloads the model. Enable a **GPU** (Colab: *Runtime › Change runtime type*; Kaggle: *Accelerator*), otherwise answers will be very slow. Locally, install Ollama yourself and then run `ollama pull qwen2.5:7b`.
For the other providers, the cell does nothing.

In [ ]:
import shutil
if LLM_PROVIDER == "ollama":
    _model = LLM_MODEL or PROVIDERS["ollama"]["default_model"]
    if IN_COLAB or IN_KAGGLE:
        if not shutil.which("ollama"):
            print("⬇️ Installing Ollama (≈1 min)…")
            subprocess.run("apt-get -qq install -y zstd pciutils > /dev/null 2>&1; "
                           "curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1", shell=True)
        subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        time.sleep(6)
    if shutil.which("ollama"):
        print(f"⬇️ Downloading model {_model} (may take a few minutes the first time)…")
        subprocess.run(["ollama", "pull", _model], stdout=subprocess.DEVNULL)
        print("✅ Ollama ready.")
    else:
        print("ℹ️ Ollama is not installed here: https://ollama.com/download then `ollama serve`.")
else:
    print(f"ℹ️ Provider “{LLM_PROVIDER}”: this cell is skipped.")

In [ ]:
try:
    llm = LLMClient(LLM_PROVIDER, LLM_MODEL or None)
    reply, latency = llm.ping()
    ui.card(f"Connected to {llm.provider.capitalize()}",
            ui.kpis([dict(label="Model", value=llm.model),
                     dict(label="Round-trip latency", value=f"{latency:.2f}", unit="s"),
                     dict(label="Test reply", value=reply[:20] or "∅")]),
            kicker="04 · The brain", highlight=True)
except Exception as e:
    llm = None
    alt = ", ".join(PROVIDERS[LLM_PROVIDER]["alternatives"])
    ui.alert(f"Could not connect to <b>{LLM_PROVIDER}</b>: <code>{str(e)[:300]}</code><br>"
             f"Check the key, the model name (alternatives: {alt}) or switch provider in section 01. "
             "Sections 05 to 07 (data and tools) work without an LLM.")

## 05 · The data: night-time lights

### What satellites measure at night

The **VIIRS Day/Night Band** sensor (Suomi-NPP satellite since 2012, then NOAA-20/21) measures night-time **radiance** in **nW/cm²/sr** at ≈ 500 m resolution. It succeeds **DMSP-OLS** (1992–2013), which was coarser and saturated over cities: the two series **cannot be joined directly** (sensor break).

| Product (Earth Engine ID) | Frequency | Period | Band used | Use in the agent |
|---|---|---|---|---|
| `NOAA/VIIRS/DNB/ANNUAL_V21` + `ANNUAL_V22` (EOG VNL v2) | Annual | 2012/13 → recent | `average_masked` (background, fires and aurorae masked) | Annual series, statistics by region |
| `NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG` | Monthly | 2014 → recent | `avg_rad`, `cf_cvg` (no. of cloud-free observations) | Monthly series, anomalies, quality |
| `NASA/VIIRS/002/VNP46A2` (Black Marble) | Daily | 2012 → recent | `DNB_BRDF_Corrected_NTL` + quality flag | Impact of an event (before/after) |
| `projects/sat-io/open-datasets/WORLDPOP/pop` (WorldPop **Global2**, R2025A) | Annual | **2015–2030** (2026–2030: projections) | `population` | Population in unlit areas, NTL per capita |
| `WorldPop/GP/100m/pop` (WorldPop GP, v1) | Annual | 2000–2014 (fallback) | `population` | Years before 2015 |
| `WM/geoLab/geoBoundaries/600/ADM{0,1,2}` | — | — | `shapeGroup` = ISO3 code | Borders and regions |

### The indicators computed

| Indicator | Definition | Interpretation |
|---|---|---|
| **SOL** (*Sum Of Lights*) | Sum of pixel radiances within an area | Proxy for total activity / electrification |
| **Mean radiance** | SOL ÷ number of pixels | Intensity, comparable across areas of different sizes |
| **Lit area** | Area of pixels > `LIT_THRESHOLD` | Spread of urbanisation / electrification |
| **Unlit population** | WorldPop population living in pixels below the threshold | (Imperfect) proxy for the electricity access gap |

### Artefacts to be aware of (and how the toolbox handles them)

| Artefact | Symptom | Countermeasure in the agent |
|---|---|---|
| **Cloud cover** | Sudden drop in a rainy-season month | `detect_anomalies` cross-checks each anomaly against `coverage_pct` |
| **Seasonality** | Regular annual cycle | Deseasonalisation before anomaly detection |
| **Gas flares** | Extremely bright points outside cities | `get_quality_report`: percentiles, maximum, alert for producer countries |
| **Blooming** | Halo of light around cities | Lighting threshold; caution with small ADM2 units |
| **Noise in dimly lit rural areas** | Unstable low values | `LIT_THRESHOLD` threshold; masked annual product |
| **Product break** | Level shift between two sources | A series uses **a single source** from start to finish (`source` parameter) |

### Connecting to Earth Engine (or demo mode)

The `DATA_BACKEND` logic:

* `gee` → Earth Engine required (explicit error otherwise);
* `demo` → **synthetic** data generated locally (for learning the mechanics only);
* `auto` → Earth Engine if a project is configured and authentication succeeds, otherwise demo.

Both “backends” expose **exactly the same methods**: the toolbox and the agent cannot tell the difference. This is good architectural practice, which would for example make it easy to add a backend based on Black Marble files downloaded from NASA Earthdata in the future.

In [ ]:
import pycountry


def country_name(iso3):
    c = pycountry.countries.get(alpha_3=str(iso3).strip().upper())
    if c is None:
        raise ValueError(f"Unknown ISO3 code: “{iso3}”. Valid examples: RWA, SEN, CIV, KEN.")
    return getattr(c, "common_name", None) or c.name


class DiskCache:
    """Very simple disk cache: avoids re-running identical Earth Engine computations (saves time and quota)."""
    def __init__(self, folder):
        self.folder = Path(folder)
        self.folder.mkdir(parents=True, exist_ok=True)

    def get(self, key_parts, compute):
        key = hashlib.md5(json.dumps(key_parts, sort_keys=True, default=str).encode()).hexdigest()
        path = self.folder / f"{key}.pkl"
        if path.exists():
            try:
                return pickle.loads(path.read_bytes())
            except Exception:
                pass
        value = compute()
        path.write_bytes(pickle.dumps(value))
        return value

    def clear(self):
        for p in self.folder.glob("*.pkl"):
            p.unlink()

CACHE = DiskCache(OUT / "cache")


def _finalize(df):
    """Adds the derived indicators from the raw sums returned by the backends."""
    df = df.copy()
    if {"sol", "n_px"} <= set(df.columns):
        df["mean_rad"] = df["sol"] / df["n_px"].replace(0, np.nan)
    if {"lit_px", "n_px"} <= set(df.columns):
        df["lit_share"] = 100 * df["lit_px"] / df["n_px"].replace(0, np.nan)
    if {"cov_px", "n_px"} <= set(df.columns):
        df["coverage_pct"] = 100 * df["cov_px"] / df["n_px"].replace(0, np.nan)
    return df.drop(columns=[c for c in ("lit_px", "cov_px") if c in df.columns])


class BaseBackend:
    """Methods shared by both backends (resolution of sources and years)."""
    synthetic = False
    PIXEL_KM2 = 0.2146   # area of a 15" VIIRS pixel at the equator

    @property
    def annual_years(self):
        return sorted({y for ys in self.availability["annual"].values() for y in ys})

    @property
    def monthly_full_years(self):
        months = self.availability["monthly_months"]
        counts = pd.Series([m[:4] for m in months]).value_counts()
        return sorted(int(y) for y, n in counts.items() if n == 12)

    def resolve_source(self, years, source="auto"):
        years = set(int(y) for y in years)
        ann, mon = set(self.annual_years), set(self.monthly_full_years)
        if source == "vnl_annual" or (source == "auto" and years <= ann):
            miss = sorted(years - ann)
            if miss:
                raise ValueError(f"Years missing from the VNL annual product: {miss}. Available: {min(ann)}–{max(ann)}. "
                                 "Try source='monthly_mean' or other years.")
            return "vnl_annual"
        miss = sorted(years - mon)
        if miss:
            allowed = sorted(ann | mon)
            raise ValueError(f"Years unavailable: {miss}. Usable years: {allowed[0]}–{allowed[-1]}.")
        return "monthly_mean"

    def source_label(self, src):
        return {"vnl_annual": "VIIRS VNL v2 annual (average_masked)",
                "monthly_mean": "Mean of VIIRS VCMSLCFG monthly composites"}.get(src, src)

    def default_year(self):
        return max(self.annual_years) if self.annual_years else max(self.monthly_full_years)

In [ ]:
NIGHT_HEX = [c.lstrip("#") for c in NIGHT]


def _ee_initialize(project):
    """Initialises Earth Engine according to the environment (Kaggle service account, Colab, local)."""
    import ee
    sa_json = get_secret("GEE_SERVICE_ACCOUNT_JSON", prompt=False)
    if sa_json:
        info = json.loads(sa_json)
        creds = ee.ServiceAccountCredentials(info["client_email"], key_data=sa_json)
        ee.Initialize(creds, project=project or info.get("project_id"))
        return ee
    try:
        ee.Initialize(project=project)
    except Exception:
        if IN_COLAB:
            ee.Authenticate()
        else:
            ee.Authenticate(auth_mode="notebook")
        ee.Initialize(project=project)
    return ee


class GEEBackend(BaseBackend):
    name = "Google Earth Engine"
    MONTHLY_ID = "NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG"
    ANNUAL_IDS = ("NOAA/VIIRS/DNB/ANNUAL_V21", "NOAA/VIIRS/DNB/ANNUAL_V22")
    DAILY_ID   = "NASA/VIIRS/002/VNP46A2"
    POP_ID     = "WorldPop/GP/100m/pop"                         # 2000–2020 (official catalogue)
    POP_G2_ID  = "projects/sat-io/open-datasets/WORLDPOP/pop"     # WorldPop Global2 R2025A, 2015–2030
    BOUNDS_ID  = "WM/geoLab/geoBoundaries/600/ADM{level}"
    SCALE      = 463.83

    def __init__(self, project):
        self.ee = _ee_initialize(project)
        self.project = project
        self.monthly = self.ee.ImageCollection(self.MONTHLY_ID)
        self._fc, self._n = {}, {}
        today = dt.date.today().isoformat()[:7]          # availability recomputed every month
        self.availability = CACHE.get(["gee-availability", today], self._availability)

    # ---------- availability ----------
    def _availability(self):
        ee = self.ee
        def years_of(cid):
            try:
                ts = ee.ImageCollection(cid).aggregate_array("system:time_start").getInfo()
                return sorted({dt.datetime.fromtimestamp(t / 1000, dt.timezone.utc).year for t in ts})
            except Exception:
                return []
        ann = {cid: years_of(cid) for cid in self.ANNUAL_IDS}
        ts = self.monthly.aggregate_array("system:time_start").getInfo()
        months = sorted({dt.datetime.fromtimestamp(t / 1000, dt.timezone.utc).strftime("%Y-%m") for t in ts})
        return {"annual": ann, "monthly_months": months}

    # ---------- boundaries ----------
    def _admin_fc(self, iso3, level):
        ee, key = self.ee, (iso3, level)
        if key in self._fc:
            return self._fc[key]
        fc = ee.FeatureCollection(self.BOUNDS_ID.format(level=level)).filter(ee.Filter.eq("shapeGroup", iso3))
        try:
            n = fc.size().getInfo()
        except Exception:
            n = 0
        if n == 0:   # fallback: FAO GAUL 2015 (filtered by country name)
            gaul = ee.FeatureCollection(f"FAO/GAUL/2015/level{level}").filter(
                ee.Filter.eq("ADM0_NAME", pycountry.countries.get(alpha_3=iso3).name))
            fc = gaul.map(lambda f: ee.Feature(f.geometry(), {
                "shapeID": ee.Number(f.get(f"ADM{level}_CODE")).format("%d"),
                "shapeName": f.get(f"ADM{level}_NAME")}))
            n = fc.size().getInfo()
            if n == 0:
                raise ValueError(f"No ADM{level} units found for {iso3} (geoBoundaries and GAUL).")
        self._fc[key] = fc.select(["shapeID", "shapeName"])
        self._n[key] = n
        return self._fc[key]

    def admin_units(self, iso3, level):
        def compute():
            fc = self._admin_fc(iso3, level)
            info = fc.map(lambda f: self.ee.Feature(None, {
                "unit_id": f.get("shapeID"), "unit": f.get("shapeName"),
                "area_km2": f.geometry().area(1000).divide(1e6)})).getInfo()
            return pd.DataFrame([x["properties"] for x in info["features"]])
        return CACHE.get(["gee-units", iso3, level], compute).copy()

    def geojson(self, iso3, level):
        def compute():
            fc = self._admin_fc(iso3, level)
            tol = 2000 if level == 0 else 800
            simp = fc.map(lambda f: self.ee.Feature(f.simplify(maxError=tol).geometry(),
                                                     {"unit_id": f.get("shapeID"), "unit": f.get("shapeName")}))
            return simp.getInfo()
        return CACHE.get(["gee-geojson", iso3, level], compute)

    # ---------- images ----------
    def _annual_rad(self, year, src):
        ee = self.ee
        if src == "vnl_annual":
            cid = next(c for c, ys in self.availability["annual"].items() if year in ys)
            img = ee.ImageCollection(cid).filterDate(f"{year}-01-01", f"{year + 1}-01-01").first()
            rad = ee.Image(img).select("average_masked").unmask(0).max(0)
        else:
            col = self.monthly.filterDate(f"{year}-01-01", f"{year + 1}-01-01").select("avg_rad")
            rad = col.map(lambda i: i.max(0)).mean().unmask(0)
            rad = rad.where(rad.lt(LIT_THRESHOLD), 0)
        return rad.rename("rad").set("year", year)

    def _stats_image(self, rad):
        ee = self.ee
        area = ee.Image.pixelArea().divide(1e6)
        lit = rad.gt(LIT_THRESHOLD)
        return ee.Image.cat([rad.rename("sol"), lit.multiply(area).rename("lit_km2"), area.rename("area_km2"),
                             lit.rename("lit_px"), ee.Image.constant(1).rename("n_px")])

    # ---------- national series ----------
    def national_series(self, iso3, start_year, end_year, frequency="annual", source="auto"):
        return CACHE.get(["gee-nat", iso3, start_year, end_year, frequency, source, LIT_THRESHOLD],
                         lambda: self._national_series(iso3, start_year, end_year, frequency, source)).copy()

    def _national_series(self, iso3, start_year, end_year, frequency, source):
        ee = self.ee
        geom = self._admin_fc(iso3, 0).geometry()
        red_kw = dict(reducer=ee.Reducer.sum(), geometry=geom, scale=self.SCALE, maxPixels=1e13, tileScale=4)
        if frequency == "annual":
            years = list(range(start_year, end_year + 1))
            src = self.resolve_source(years, source)
            col = ee.ImageCollection.fromImages([self._annual_rad(y, src).set("period", str(y)) for y in years])
            def red(img):
                d = self._stats_image(ee.Image(img).select("rad")).reduceRegion(**red_kw)
                return ee.Feature(None, d).set("period", img.get("period"))
            label = self.source_label(src)
        else:
            col = self.monthly.filterDate(f"{start_year}-01-01", f"{end_year + 1}-01-01")
            thr = LIT_THRESHOLD
            def red(img):
                rad = img.select("avg_rad").max(0).unmask(0)
                rad = rad.where(rad.lt(thr), 0).rename("rad")
                st = self._stats_image(rad).addBands(img.select("cf_cvg").gt(0).unmask(0).rename("cov_px"))
                return ee.Feature(None, st.reduceRegion(**red_kw)).set("period", img.date().format("YYYY-MM"))
            label = "VIIRS VCMSLCFG monthly (avg_rad, threshold applied)"
        rows = [f["properties"] for f in ee.FeatureCollection(col.map(red)).getInfo()["features"]]
        df = _finalize(pd.DataFrame(rows)).sort_values("period").reset_index(drop=True)
        df["source"] = label
        return df

    # ---------- zonal statistics ----------
    def zonal_stats(self, iso3, level, years, source="auto"):
        years = sorted(int(y) for y in years)
        return CACHE.get(["gee-zonal", iso3, level, years, source, LIT_THRESHOLD],
                         lambda: self._zonal_stats(iso3, level, years, source)).copy()

    def _zonal_stats(self, iso3, level, years, source):
        ee = self.ee
        src = self.resolve_source(years, source)
        fc = self._admin_fc(iso3, level)
        chunk = max(1, 4000 // max(1, self._n[(iso3, level)]))
        rows = []
        for i in range(0, len(years), chunk):
            imgs = [self._annual_rad(y, src) for y in years[i:i + chunk]]
            def per_img(img):
                img = ee.Image(img)
                y = img.get("year")
                red = self._stats_image(img.select("rad")).reduceRegions(
                    collection=fc, reducer=ee.Reducer.sum(), scale=self.SCALE, tileScale=4)
                return red.map(lambda f: ee.Feature(None, f.toDictionary()).set("year", y))
            res = ee.FeatureCollection(ee.ImageCollection.fromImages(imgs).map(per_img)).flatten().getInfo()
            rows += [f["properties"] for f in res["features"]]
        df = pd.DataFrame(rows).rename(columns={"shapeID": "unit_id", "shapeName": "unit"})
        df = _finalize(df)
        df["year"] = df["year"].astype(int)
        df["source"] = self.source_label(src)
        return df

    # ---------- population ----------
    def population_stats(self, iso3, level, year, source="auto"):
        return CACHE.get(["gee-pop-g2", iso3, level, year, source, LIT_THRESHOLD],
                         lambda: self._population_stats(iso3, level, year, source)).copy()

    def _worldpop(self, iso3, year):
        """WorldPop Global2 (2015–2030) first, WorldPop GP v1 (2000–2020) as fallback.
        Returns (population image, year used, source label)."""
        ee = self.ee
        geom = self._admin_fc(iso3, 0).geometry()
        y = int(min(max(year, 2015), 2030))
        if year >= 2015:
            try:
                col = (ee.ImageCollection(self.POP_G2_ID).filterDate(f"{y}-01-01", f"{y + 1}-01-01")
                       .filterBounds(geom))
                if col.size().getInfo() > 0:
                    proj = col.first().select(0).projection()
                    img = col.mosaic().select(0).rename("population").setDefaultProjection(proj)
                    return img, y, "WorldPop Global2 R2025A" + (" (projection)" if y > 2025 else "")
            except Exception:
                pass
        y = int(min(max(year, 2000), 2020))
        col = ee.ImageCollection(self.POP_ID).filter(ee.Filter.eq("country", iso3)).filter(ee.Filter.eq("year", y))
        if col.size().getInfo() == 0:
            raise ValueError(f"WorldPop unavailable for {iso3} in {year}.")
        return col.first().select("population"), y, "WorldPop GP v1"

    def _population_stats(self, iso3, level, year, source):
        ee = self.ee
        rad = self._annual_rad(year, self.resolve_source([year], source))
        pop, pop_year, pop_source = self._worldpop(iso3, year)
        area = float(self.admin_units(iso3, 0)["area_km2"].sum())
        if area < 300_000:                       # native resolution (100 m)
            pop_img, scale = pop, 100
        else:                                    # large countries: density × area (avoids computation timeouts)
            density = pop.divide(ee.Image.pixelArea().reproject(pop.projection()))
            pop_img, scale = density.multiply(ee.Image.pixelArea()), 250
        img = ee.Image.cat([pop_img.rename("pop"), pop_img.multiply(rad.lt(LIT_THRESHOLD)).rename("pop_unlit")])
        res = img.reduceRegions(collection=self._admin_fc(iso3, level), reducer=ee.Reducer.sum(),
                                scale=scale, tileScale=16).map(lambda f: ee.Feature(None, f.toDictionary())).getInfo()
        df = pd.DataFrame([f["properties"] for f in res["features"]]).rename(
            columns={"shapeID": "unit_id", "shapeName": "unit"})
        df["pop_year"] = pop_year
        df["pop_source"] = pop_source
        return df

    # ---------- event (daily Black Marble) ----------
    def event_impact(self, iso3, level, event_date, days_before=30, days_after=30):
        return CACHE.get(["gee-event", iso3, level, event_date, days_before, days_after],
                         lambda: self._event_impact(iso3, level, event_date, days_before, days_after)).copy()

    def _event_impact(self, iso3, level, event_date, days_before, days_after):
        ee = self.ee
        d = ee.Date(event_date)
        col = ee.ImageCollection(self.DAILY_ID)
        def clean(img):
            q = img.select("Mandatory_Quality_Flag")
            return img.select("DNB_BRDF_Corrected_NTL").updateMask(q.lte(1)).rename("rad")
        before = col.filterDate(d.advance(-days_before, "day"), d).map(clean)
        after = col.filterDate(d, d.advance(days_after, "day")).map(clean)
        n_b, n_a = before.size().getInfo(), after.size().getInfo()
        if n_b == 0 or n_a == 0:
            raise ValueError(f"No Black Marble images in the window (before: {n_b}, after: {n_a}). "
                             "The date may be too recent or earlier than 2012.")
        img = ee.Image.cat([before.median().unmask(0).rename("sol_before"), after.median().unmask(0).rename("sol_after"),
                            before.count().gt(0).unmask(0).rename("cov_b"), after.count().gt(0).unmask(0).rename("cov_a"),
                            ee.Image.constant(1).rename("n_px")])
        res = img.reduceRegions(collection=self._admin_fc(iso3, level), reducer=ee.Reducer.sum(),
                                scale=500, tileScale=4).map(lambda f: ee.Feature(None, f.toDictionary())).getInfo()
        df = pd.DataFrame([f["properties"] for f in res["features"]]).rename(
            columns={"shapeID": "unit_id", "shapeName": "unit"})
        df["coverage_before_pct"] = 100 * df.pop("cov_b") / df["n_px"]
        df["coverage_after_pct"] = 100 * df.pop("cov_a") / df["n_px"]
        df["n_images_before"], df["n_images_after"] = n_b, n_a
        return df.drop(columns="n_px")

    # ---------- quality ----------
    def quality_report(self, iso3, year, source="auto"):
        return CACHE.get(["gee-quality", iso3, year, source, LIT_THRESHOLD],
                         lambda: self._quality_report(iso3, year, source))

    def _quality_report(self, iso3, year, source):
        ee = self.ee
        geom = self._admin_fc(iso3, 0).geometry()
        col = self.monthly.filterDate(f"{year}-01-01", f"{year + 1}-01-01")
        def per(img):
            cov = img.select("cf_cvg")
            d = ee.Image.cat([cov.gt(0).rename("covered"), cov.rename("cf_cvg")]).reduceRegion(
                ee.Reducer.mean(), geom, 2000, maxPixels=1e13, bestEffort=True)
            return ee.Feature(None, d).set("period", img.date().format("YYYY-MM"))
        cov = pd.DataFrame([f["properties"] for f in ee.FeatureCollection(col.map(per)).getInfo()["features"]])
        cov = cov.rename(columns={"covered": "coverage_share", "cf_cvg": "mean_cloudfree_obs"})
        cov["coverage_pct"] = 100 * cov.pop("coverage_share")
        rad = self._annual_rad(year, self.resolve_source([year], source))
        pct = rad.updateMask(rad.gt(LIT_THRESHOLD)).reduceRegion(
            ee.Reducer.percentile([50, 90, 99]).combine(ee.Reducer.max(), "", True),
            geom, self.SCALE, maxPixels=1e13, bestEffort=True).getInfo()
        return {"monthly_coverage": cov.sort_values("period"),
                "percentiles": {k.replace("rad_", ""): v for k, v in pct.items()}}

    # ---------- image ----------
    def thumbnail(self, iso3, year, source="auto"):
        def compute():
            geom = self._admin_fc(iso3, 0).geometry()
            rad = self._annual_rad(year, self.resolve_source([year], source))
            vis = rad.add(0.1).log10().clip(geom).visualize(min=-1, max=2, palette=NIGHT_HEX)
            url = vis.getThumbURL({"region": geom.bounds(), "dimensions": 1000, "format": "png"})
            return requests.get(url, timeout=180).content
        return CACHE.get(["gee-thumb", iso3, year, source], compute)

In [ ]:
from shapely.geometry import shape, box, mapping
import shapely


class DemoBackend(BaseBackend):
    """SYNTHETIC data, deterministic for a given ISO code. Real boundaries (geoBoundaries) if the Internet connection
    allows, otherwise a fictitious grid. Used only to learn the agent's mechanics — never quote these figures."""
    name = "Demo mode (synthetic data)"
    synthetic = True

    def __init__(self):
        months = [f"{y}-{m:02d}" for y in range(2014, 2026) for m in range(1, 13)][:138]   # 2014-01 → 2025-06
        self.availability = {"annual": {"DEMO_ANNUAL": list(range(2014, 2025))}, "monthly_months": months}
        self._units = {}

    # ---------- boundaries ----------
    def _fetch_geoboundaries(self, iso3, level):
        try:
            meta = requests.get(f"https://www.geoboundaries.org/api/current/gbOpen/{iso3}/ADM{level}/", timeout=15).json()
            gj = requests.get(meta["simplifiedGeometryGeoJSON"], timeout=60).json()
            feats = []
            for i, f in enumerate(gj["features"]):
                p = f.get("properties", {})
                feats.append({"type": "Feature", "geometry": f["geometry"],
                              "properties": {"unit_id": p.get("shapeID", f"{iso3}-{level}-{i}"),
                                             "unit": p.get("shapeName", f"Unit {i + 1}")}})
            return {"type": "FeatureCollection", "features": feats}
        except Exception:
            return None

    def _synthetic_grid(self, iso3, level):
        rng = np.random.default_rng(zlib.crc32(f"{iso3}-grid".encode()))
        nx, ny = (3, 2) if level == 0 else ((4, 3) if level == 1 else (8, 6))
        if level == 0:
            nx = ny = 1
        x0, y0, w, h = 28.8, -2.9, 2.2, 1.9
        feats = []
        for j in range(ny):
            for i in range(nx):
                k = j * nx + i
                g = box(x0 + i * w / nx, y0 + j * h / ny, x0 + (i + 1) * w / nx, y0 + (j + 1) * h / ny)
                name = country_name(iso3) if level == 0 else f"Region {chr(65 + k)}" if level == 1 else f"District {k + 1:02d}"
                feats.append({"type": "Feature", "geometry": mapping(g),
                              "properties": {"unit_id": f"{iso3}-ADM{level}-{k + 1:03d}", "unit": name}})
        return {"type": "FeatureCollection", "features": feats}

    def geojson(self, iso3, level):
        return CACHE.get(["demo-geojson", iso3, level],
                         lambda: self._fetch_geoboundaries(iso3, level) or self._synthetic_grid(iso3, level))

    def _unit_table(self, iso3, level):
        key = (iso3, level)
        if key in self._units:
            return self._units[key]
        gj = self.geojson(iso3, level)
        rows = []
        for f in gj["features"]:
            g = shape(f["geometry"])
            c = g.centroid
            area = g.area * 111.32 ** 2 * math.cos(math.radians(c.y))
            rows.append(dict(unit_id=f["properties"]["unit_id"], unit=f["properties"]["unit"],
                             area_km2=area, lon=c.x, lat=c.y))
        df = pd.DataFrame(rows)
        rng = np.random.default_rng(zlib.crc32(f"{iso3}-{level}".encode()))
        n = len(df)
        df["base_rad"] = rng.lognormal(mean=np.log(0.35), sigma=0.8, size=n)          # mean radiance in 2014
        capital = int(rng.integers(n)) if n > 1 else 0
        df.loc[capital, "base_rad"] = df["base_rad"].max() * 6 + 3
        df["growth"] = np.clip(rng.normal(0.055, 0.035, n), -0.03, 0.16)
        df["pop_density"] = rng.lognormal(np.log(180), 0.6, n) * (1 + 3 * df["base_rad"] / df["base_rad"].max())
        df["is_capital"] = False
        df.loc[capital, "is_capital"] = True
        self._units[key] = df
        return df

    @staticmethod
    def _noise(*parts, sigma=0.03):
        return float(np.random.default_rng(zlib.crc32("|".join(map(str, parts)).encode())).normal(1, sigma))

    def _unit_rad(self, iso3, u, year, month=None):
        rad = u.base_rad * (1 + u.growth) ** (year - 2014) * self._noise(iso3, u.unit_id, year)
        if year == 2020:
            rad *= 0.95                                   # synthetic “2020” shock
        if month is not None:
            rad *= 1 + 0.06 * math.cos(2 * math.pi * (month - 12) / 12)   # seasonality: December brighter
            rad *= self._noise(iso3, u.unit_id, year, month, sigma=0.04)
        return rad

    def _coverage(self, iso3, year, month):
        rainy = month in (4, 5, 11)
        cov = (0.62 if rainy else 0.95) * self._noise(iso3, "cov", year, month, sigma=0.05)
        return float(np.clip(cov, 0.3, 1.0))

    def _row(self, iso3, u, rad, cov=None):
        n_px = u.area_km2 / self.PIXEL_KM2
        lit_share = float(np.clip(0.02 + 0.9 * (1 - math.exp(-rad / 1.2)), 0, 0.98))
        r = dict(sol=rad * n_px, n_px=n_px, area_km2=u.area_km2, lit_km2=u.area_km2 * lit_share, lit_px=n_px * lit_share)
        if cov is not None:
            r["cov_px"] = n_px * cov
        return r

    # ---------- common API ----------
    def admin_units(self, iso3, level):
        return self._unit_table(iso3, level)[["unit_id", "unit", "area_km2"]].copy()

    def national_series(self, iso3, start_year, end_year, frequency="annual", source="auto"):
        units = self._unit_table(iso3, 1)
        rows = []
        if frequency == "annual":
            src = self.resolve_source(range(start_year, end_year + 1), source)
            for y in range(start_year, end_year + 1):
                parts = [self._row(iso3, u, self._unit_rad(iso3, u, y)) for u in units.itertuples()]
                rows.append({"period": str(y), **pd.DataFrame(parts).sum().to_dict()})
            label = self.source_label(src)
        else:
            outage = (2023, 3)                              # real synthetic drop (good coverage)
            for m_str in self.availability["monthly_months"]:
                y, m = int(m_str[:4]), int(m_str[5:])
                if not start_year <= y <= end_year:
                    continue
                cov = self._coverage(iso3, y, m)
                factor = (0.72 if (y, m) == outage else 1.0) * (0.8 + 0.2 * cov)   # clouds → underestimated radiance
                parts = [self._row(iso3, u, self._unit_rad(iso3, u, y, m) * factor, cov) for u in units.itertuples()]
                rows.append({"period": m_str, **pd.DataFrame(parts).sum().to_dict()})
            label = "Synthetic monthly"
        df = _finalize(pd.DataFrame(rows).drop(columns=["area_km2"]))
        df["source"] = label + " — DEMO"
        return df

    def zonal_stats(self, iso3, level, years, source="auto"):
        src = self.resolve_source(years, source)
        units = self._unit_table(iso3, level)
        rows = []
        for y in sorted(int(v) for v in years):
            for u in units.itertuples():
                rows.append(dict(year=y, unit_id=u.unit_id, unit=u.unit, **self._row(iso3, u, self._unit_rad(iso3, u, y))))
        df = _finalize(pd.DataFrame(rows))
        df["source"] = self.source_label(src) + " — DEMO"
        return df

    def population_stats(self, iso3, level, year, source="auto"):
        self.resolve_source([year], source)
        pop_year = int(min(max(year, 2000), 2030))
        rows = []
        for u in self._unit_table(iso3, level).itertuples():
            pop = u.area_km2 * u.pop_density * (1.026 ** (pop_year - 2014))
            rad = self._unit_rad(iso3, u, year)
            unlit_share = float(np.clip(0.92 * math.exp(-rad / 0.9), 0.02, 0.97))
            rows.append(dict(unit_id=u.unit_id, unit=u.unit, pop=pop, pop_unlit=pop * unlit_share))
        df = pd.DataFrame(rows)
        df["pop_year"] = pop_year
        df["pop_source"] = "WorldPop (DEMO)"
        return df

    def event_impact(self, iso3, level, event_date, days_before=30, days_after=30):
        d = pd.Timestamp(event_date)
        units = self._unit_table(iso3, level)
        hit = int(np.random.default_rng(zlib.crc32(f"{iso3}{event_date}".encode())).integers(len(units)))
        rows = []
        for i, u in enumerate(units.itertuples()):
            before = self._row(iso3, u, self._unit_rad(iso3, u, d.year, d.month))["sol"]
            change = -0.42 if i == hit else np.random.default_rng(zlib.crc32(f"{u.unit_id}{event_date}".encode())).normal(-0.02, 0.04)
            rows.append(dict(unit_id=u.unit_id, unit=u.unit, sol_before=before, sol_after=before * (1 + change),
                             coverage_before_pct=88.0, coverage_after_pct=84.0,
                             n_images_before=days_before, n_images_after=days_after))
        return pd.DataFrame(rows)

    def quality_report(self, iso3, year, source="auto"):
        self.resolve_source([year], source)
        cov = pd.DataFrame([dict(period=f"{year}-{m:02d}", coverage_pct=100 * self._coverage(iso3, year, m),
                                 mean_cloudfree_obs=round(12 * self._coverage(iso3, year, m), 1)) for m in range(1, 13)])
        units = self._unit_table(iso3, 1)
        top = float(units.base_rad.max() * (1 + units.growth.mean()) ** (year - 2014))
        return {"monthly_coverage": cov, "percentiles": {"p50": 1.4, "p90": 6.8, "p99": round(top * 2.5, 1),
                                                          "max": round(top * 6, 1)}}

    def thumbnail(self, iso3, year, source="auto"):
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        from matplotlib.colors import LinearSegmentedColormap
        units = self._unit_table(iso3, 1)
        gj = self.geojson(iso3, 0)
        country = shapely.unary_union([shape(f["geometry"]) for f in gj["features"]])
        minx, miny, maxx, maxy = country.bounds
        nx = 700
        ny = max(50, int(nx * (maxy - miny) / (maxx - minx)))
        xs, ys = np.meshgrid(np.linspace(minx, maxx, nx), np.linspace(maxy, miny, ny))
        field = np.full(xs.shape, 0.05)
        rng = np.random.default_rng(zlib.crc32(f"{iso3}-thumb".encode()))
        span = max(maxx - minx, maxy - miny)
        for u in units.itertuples():
            amp = self._unit_rad(iso3, u, year) * 8
            sig = span * (0.012 + 0.02 * u.is_capital)
            field += amp * np.exp(-((xs - u.lon) ** 2 + (ys - u.lat) ** 2) / (2 * sig ** 2))
            for _ in range(12):                        # secondary towns / villages
                cx, cy = u.lon + rng.normal(0, span * 0.06), u.lat + rng.normal(0, span * 0.06)
                field += amp * 0.08 * rng.random() * np.exp(-((xs - cx) ** 2 + (ys - cy) ** 2) / (2 * (span * 0.004) ** 2))
        inside = shapely.contains_xy(country, xs, ys)
        img = np.log10(field + 0.1)
        img = np.where(inside, img, np.nan)
        cmap = LinearSegmentedColormap.from_list("night", NIGHT)
        cmap.set_bad(alpha=0)
        fig, ax = plt.subplots(figsize=(8, 8 * ny / nx), dpi=120)
        ax.imshow(img, cmap=cmap, vmin=-1, vmax=2, interpolation="bilinear")
        ax.axis("off")
        buf = io.BytesIO()
        fig.savefig(buf, format="png", bbox_inches="tight", pad_inches=0, transparent=True)
        plt.close(fig)
        return buf.getvalue()

In [ ]:
def connect_backend(mode=DATA_BACKEND, project=GEE_PROJECT):
    project = (project or get_secret("GEE_PROJECT", prompt=False) or "").strip()
    if mode in ("gee", "auto"):
        if not project and mode == "auto":
            ui.warn("No Earth Engine project (<code>GEE_PROJECT</code>): switching to <b>demo mode</b> (synthetic data).")
            return DemoBackend()
        try:
            return GEEBackend(project)
        except Exception as e:
            if mode == "gee":
                raise
            ui.warn(f"Earth Engine unavailable (<code>{str(e)[:200]}</code>): switching to <b>demo mode</b>.")
    return DemoBackend()


country_name(COUNTRY_ISO3)            # validates the ISO3 code right away
backend = connect_backend()
ann = backend.annual_years
mon = backend.availability["monthly_months"]
body = ui.kpis([
    dict(label="Country", value=COUNTRY_ISO3, unit=country_name(COUNTRY_ISO3)),
    dict(label="Annual available", value=f"{min(ann)}–{max(ann)}" if ann else "—"),
    dict(label="Monthly available", value=f"{mon[0]} → {mon[-1]}"),
    dict(label="“Lit” threshold", value=LIT_THRESHOLD, unit="nW/cm²/sr"),
])
if backend.synthetic:
    body += ('<div class="ntl-alert"><b>DEMO MODE — SYNTHETIC DATA.</b> Perfect for learning the mechanics of '
             "the agent; the values do not describe reality and must never be quoted. "
             "Set <code>GEE_PROJECT</code> to use real data.</div>")
ui.card(backend.name, body, kicker="05 · Data source", highlight=not backend.synthetic)

## 06 · The agent's toolbox

This is the heart of the work. An LLM can only use a tool if it **understands what it is for** and **how to call it**. Five principles guide the design:

1. **The Python signature *is* the contract.** The `@tool` decorator reads the types (`int`, `Literal[...]`, `Optional`) and the docstring to automatically generate the **JSON schema** sent to the LLM. A good docstring = an agent that makes good choices.
2. **The LLM sees summaries, Python keeps the data.** Each computation tool stores its full table in the session under an identifier (`ds03_admin_stats`) and returns only a compact summary to the LLM. The plotting tools reuse this identifier: the context stays short, and the charts are built from the exact data.
3. **Errors are information.** An exception is returned to the LLM as a clear message (“available years: 2014–2024”) so that it can **correct its own** arguments.
4. **Safe default values.** Years, administrative level, country: everything has a sensible default, which limits errors from small models.
5. **Caution is built in.** The tools themselves flag limitations (cloud cover, source break, small sample, synthetic data).

In [ ]:
TOOLS = {}                      # name -> dict(fn, family, core, schema)


def _py_to_schema(ann):
    origin = get_origin(ann)
    if origin is Union:                                        # Optional[X]
        args = [a for a in get_args(ann) if a is not type(None)]
        return _py_to_schema(args[0])
    if origin is Literal:
        vals = list(get_args(ann))
        return {"type": "string", "enum": [str(v) for v in vals]}
    return {"type": {int: "integer", float: "number", bool: "boolean", str: "string"}.get(ann, "string")}


def _parse_doc(doc):
    doc = inspect.cleandoc(doc or "")
    desc, _, args_part = doc.partition("Args:")
    params = {}
    for line in args_part.splitlines():
        m = re.match(r"\s*(\w+)\s*:\s*(.+)", line)
        if m:
            params[m.group(1)] = m.group(2).strip()
    return " ".join(desc.split()), params


def tool(family, core=False):
    """Decorator: registers a Python function as a tool and generates its JSON schema."""
    def deco(fn):
        desc, pdocs = _parse_doc(fn.__doc__)
        props, required = {}, []
        for name, p in inspect.signature(fn).parameters.items():
            sch = _py_to_schema(p.annotation)
            d = pdocs.get(name, "")
            if p.default is inspect.Parameter.empty:
                required.append(name)
            elif p.default is not None:
                d += f" (default: {p.default})"
            sch["description"] = d
            props[name] = sch
        TOOLS[fn.__name__] = dict(fn=fn, family=family, core=core, schema={
            "type": "function",
            "function": {"name": fn.__name__, "description": desc,
                         "parameters": {"type": "object", "properties": props, "required": required}}})
        return fn
    return deco


class Session:
    """Working memory: datasets, figures, key indicators and turn history."""
    def __init__(self):
        self.datasets, self.figures, self.turns = {}, {}, []
        self.kpis, self.turn_id = [], 0

    def new_turn(self):
        self.turn_id += 1
        self.kpis = []

    def add_dataset(self, df, kind, title, **meta):
        ds_id = f"ds{len(self.datasets) + 1:02d}_{kind}"
        self.datasets[ds_id] = dict(df=df.reset_index(drop=True), kind=kind, title=title, meta=meta, turn=self.turn_id)
        return ds_id

    def get(self, ds_id):
        if ds_id not in self.datasets:
            raise KeyError(f"Unknown dataset: {ds_id}. Available: {list(self.datasets)[-8:]}")
        return self.datasets[ds_id]

    def add_figure(self, obj, title, kind="plotly"):
        fig_id = f"fig{len(self.figures) + 1:02d}"
        self.figures[fig_id] = dict(obj=obj, title=title, kind=kind, turn=self.turn_id)
        if kind == "plotly":
            try:
                obj.write_html(OUT / "figures" / f"{fig_id}.html", include_plotlyjs="cdn")
            except Exception:
                pass
        else:
            (OUT / "figures" / f"{fig_id}.png").write_bytes(obj)
        return fig_id

    def add_kpi(self, label, value, unit="", delta=None):
        self.kpis.append(dict(label=label, value=value, unit=unit, delta=delta))

S = Session()

# ---------- shared utilities ----------
LABELS = {
    "sol": "Sum of lights (SOL)", "mean_rad": "Mean radiance (nW/cm²/sr)",
    "lit_km2": "Lit area (km²)", "lit_share": "Share of lit area (%)",
    "sol_share": "Share of national SOL (%)", "sol_density": "SOL per km²",
    "pct_change": "Change (%)", "cagr_pct": "Compound annual growth rate (%)",
    "pop": "Population", "pop_unlit": "Population in unlit areas",
    "pop_unlit_share": "Population in unlit areas (%)", "sol_per_1000": "SOL per 1,000 inhabitants",
    "coverage_pct": "Cloud-free coverage (%)", "indicator": "Indicator", "sol_before": "SOL before",
    "sol_after": "SOL after", "lit_km2_change": "Change in lit area (km²)",
    "sol_per_capita_1000": "SOL per 1,000 inhabitants", "electricity_access": "Access to electricity (%)",
}


def _iso(iso3=None):
    iso = (iso3 or COUNTRY_ISO3).strip().upper()
    country_name(iso)
    return iso


def _level(level):
    level = int(level)
    if level not in (1, 2):
        raise ValueError("The administrative level must be 1 (regions/provinces) or 2 (districts).")
    return level


def _years(start_year=None, end_year=None, default_span=10):
    last = backend.default_year()
    end = int(end_year) if end_year else last
    start = int(start_year) if start_year else max(end - default_span + 1, min(backend.annual_years + backend.monthly_full_years))
    if start > end:
        start, end = end, start
    return start, end


def _records(df, n=12, cols=None):
    d = df[cols] if cols else df
    d = d.head(n).copy()
    for c in d.select_dtypes("number").columns:
        d[c] = d[c].map(lambda v: None if pd.isna(v) else (round(float(v), 2) if abs(v) < 1e5 else round(float(v))))
    return d.to_dict("records")


def _cagr(v0, v1, n_years):
    if v0 is None or v0 <= 0 or n_years <= 0:
        return None
    return 100 * ((v1 / v0) ** (1 / n_years) - 1)


def _trend(x, y):
    from scipy import stats
    x, y = np.asarray(x, float), np.asarray(y, float)
    if len(x) < 3:
        return {}
    r = stats.linregress(x, y)
    return dict(slope=r.slope, slope_pct_of_mean=100 * r.slope / np.mean(y), r2=r.rvalue ** 2, p_value=r.pvalue)


def _gini(v):
    v = np.sort(np.asarray(v, float))
    v = v[v >= 0]
    if len(v) == 0 or v.sum() == 0:
        return None
    n = len(v)
    return float((2 * np.arange(1, n + 1) - n - 1).dot(v) / (n * v.sum()))


def _match_units(names, candidates):
    """Matches names typed by the user to the official names (tolerates accents and typos)."""
    out = []
    for n in names:
        m = difflib.get_close_matches(n.strip(), list(candidates), n=1, cutoff=0.5)
        if m:
            out.append(m[0])
    return out


def _base_notes():
    notes = []
    if backend.synthetic:
        notes.append("SYNTHETIC DATA (demo mode): do not cite these figures as real.")
    return notes

### 6.1 · Discovery: framing the analysis

In [ ]:
@tool("Discovery", core=True)
def get_country_profile(iso3: Optional[str] = None) -> dict:
    """Country profile: name, number of ADM1 and ADM2 administrative units, area, data source and available
    years. Call it first to frame an analysis or to find out which years can be used.

    Args:
        iso3: ISO 3166-1 alpha-3 country code (e.g. RWA, SEN). Default: the session country.
    """
    iso = _iso(iso3)
    adm1 = backend.admin_units(iso, 1)
    try:
        n_adm2 = len(backend.admin_units(iso, 2))
    except Exception:
        n_adm2 = None
    area = float(backend.admin_units(iso, 0)["area_km2"].sum())
    S.add_kpi("Area", fmt(area), "km²")
    S.add_kpi("ADM1 units", len(adm1))
    return {"iso3": iso, "country": country_name(iso), "area_km2": round(area), "n_adm1": len(adm1),
            "n_adm2": n_adm2, "adm1_examples": adm1["unit"].head(8).tolist(),
            "data_backend": backend.name, "annual_years": f"{min(backend.annual_years)}–{max(backend.annual_years)}",
            "monthly_range": f"{backend.availability['monthly_months'][0]} → {backend.availability['monthly_months'][-1]}",
            "lit_threshold_nw": LIT_THRESHOLD, "notes": _base_notes()}


@tool("Discovery")
def get_data_availability() -> dict:
    """Describes the available data products (annual, monthly, daily, population), their periods, the
    definitions of the indicators (SOL, mean radiance, lit area) and the main limitations."""
    return {"products": {
                "annual": {"years": backend.annual_years, "source": backend.source_label("vnl_annual")},
                "monthly": {"first": backend.availability["monthly_months"][0],
                            "last": backend.availability["monthly_months"][-1],
                            "complete_years": backend.monthly_full_years},
                "daily_black_marble": "2012 → a few days/weeks before today (assess_event_impact tool)",
                "population_worldpop": "2015–2030: WorldPop Global2 R2025A (2026–2030 = projections); 2000–2014: WorldPop GP v1"},
            "definitions": {"sol": "Sum of radiances (nW/cm²/sr) of the pixels in the area",
                            "mean_rad": "SOL / number of pixels", "lit_km2": f"Area of pixels > {LIT_THRESHOLD} nW/cm²/sr",
                            "lit_share": "Share of lit area (%)"},
            "caveats": ["NTL = proxy for activity and electrification, not a direct measure of GDP or poverty",
                        "Clouds, seasonality, gas flares, urban halo (blooming), rural noise",
                        "Do not mix two products in the same series (level break)"],
            "notes": _base_notes()}


@tool("Discovery", core=True)
def list_admin_units(level: int = 1, iso3: Optional[str] = None) -> dict:
    """Lists the country's administrative units (ADM1 regions or ADM2 districts) with their area.
    Useful to check the exact spelling of a region before a targeted analysis.

    Args:
        level: Administrative level: 1 = regions/provinces, 2 = districts.
        iso3: ISO3 country code. Default: the session country.
    """
    iso, level = _iso(iso3), _level(level)
    df = backend.admin_units(iso, level).sort_values("area_km2", ascending=False)
    ds = S.add_dataset(df, "admin_units", f"ADM{level} units — {iso}", iso3=iso, level=level)
    return {"dataset_id": ds, "count": len(df), "units": _records(df, 60, ["unit", "area_km2"]), "notes": _base_notes()}

### 6.2 · Time series and anomalies

`detect_anomalies` illustrates a good practice: **never interpret a drop in brightness without looking at cloud cover**. The method is deliberately simple and explainable: deseasonalisation using the median monthly index, trend via a rolling median, then a robust z-score (MAD).

In [ ]:
@tool("Time series", core=True)
def get_national_timeseries(start_year: Optional[int] = None, end_year: Optional[int] = None,
                            frequency: Literal["annual", "monthly"] = "annual",
                            source: Literal["auto", "vnl_annual", "monthly_mean"] = "auto",
                            iso3: Optional[str] = None) -> dict:
    """NATIONAL night-time lights time series: SOL, mean radiance, lit area (and cloud cover for
    monthly data). Computes the total change, the compound annual growth rate (CAGR) and the linear
    trend. Returns a dataset_id to pass to plot_timeseries.

    Args:
        start_year: First year (default: 10 years before the latest available year, 3 years for monthly).
        end_year: Last year (default: latest available year).
        frequency: 'annual' or 'monthly'.
        source: Product for annual data: 'auto', 'vnl_annual' (masked VNL) or 'monthly_mean' (mean of monthly composites).
        iso3: ISO3 country code. Default: the session country.
    """
    iso = _iso(iso3)
    start, end = _years(start_year, end_year, default_span=3 if frequency == "monthly" else 10)
    df = backend.national_series(iso, start, end, frequency, source)
    if df.empty:
        raise ValueError("No data for this period.")
    ds = S.add_dataset(df, f"national_{frequency}", f"National {frequency} series — {iso}", iso3=iso, frequency=frequency)
    first, last = df.iloc[0], df.iloc[-1]
    summary = {"period": f"{first.period} → {last.period}", "n_points": len(df),
               "sol_first": first.sol, "sol_last": last.sol,
               "sol_change_pct": 100 * (last.sol / first.sol - 1) if first.sol else None,
               "lit_km2_first": first.lit_km2, "lit_km2_last": last.lit_km2,
               "peak_period": df.loc[df.sol.idxmax(), "period"], "lowest_period": df.loc[df.sol.idxmin(), "period"]}
    notes = _base_notes() + [f"Source: {df['source'].iloc[0]}"]
    if frequency == "annual":
        n = int(last.period) - int(first.period)
        summary["cagr_pct"] = _cagr(first.sol, last.sol, n)
        tr = _trend(df.period.astype(int), df.sol)
        if tr:
            summary["trend_pct_per_year"] = tr["slope_pct_of_mean"]
            summary["trend_r2"], summary["trend_p_value"] = tr["r2"], tr["p_value"]
        S.add_kpi(f"SOL {last.period}", fmt(last.sol), "", summary["sol_change_pct"])
        if summary.get("cagr_pct") is not None:
            S.add_kpi("SOL CAGR", f"{summary['cagr_pct']:.1f}", "%/yr")
    else:
        low = df[df.coverage_pct < 70]
        summary["mean_coverage_pct"] = df.coverage_pct.mean()
        if len(low):
            notes.append(f"{len(low)} months with cloud-free coverage < 70%: interpret these values with caution "
                         f"({', '.join(low.period.head(8))}).")
        S.add_kpi("Mean coverage", f"{df.coverage_pct.mean():.0f}", "%")
    S.add_kpi("Lit area", fmt(last.lit_km2), "km²",
              100 * (last.lit_km2 / first.lit_km2 - 1) if first.lit_km2 else None)
    cols = ["period", "sol", "mean_rad", "lit_km2", "lit_share"] + (["coverage_pct"] if frequency == "monthly" else [])
    return {"dataset_id": ds, "summary": summary, "data": _records(df, 40 if frequency == "annual" else 14, cols),
            "notes": notes}


@tool("Time series")
def get_admin_timeseries(level: int = 1, start_year: Optional[int] = None, end_year: Optional[int] = None,
                         units: Optional[str] = None, iso3: Optional[str] = None) -> dict:
    """Annual series BY REGION (panel): SOL, mean radiance and lit area of each administrative unit,
    with the CAGR of each. Allows tracking one or more named regions. Returns a dataset_id for
    plot_timeseries (one line per region).

    Args:
        level: Administrative level: 1 = regions, 2 = districts.
        start_year: First year (default: 10 years before the latest available one).
        end_year: Last year (default: latest available).
        units: Comma-separated region names to filter on (e.g. 'Kigali, Southern'). Empty = all.
        iso3: ISO3 country code. Default: the session country.
    """
    iso, level = _iso(iso3), _level(level)
    start, end = _years(start_year, end_year)
    df = backend.zonal_stats(iso, level, list(range(start, end + 1)))
    if units:
        keep = _match_units(units.split(","), df.unit.unique())
        if not keep:
            raise ValueError(f"No recognised region in “{units}”. Use list_admin_units for the exact names.")
        df = df[df.unit.isin(keep)]
    df = df.sort_values(["unit", "year"])
    ds = S.add_dataset(df, "admin_timeseries", f"ADM{level} series {start}–{end} — {iso}", iso3=iso, level=level)
    g = df.groupby("unit").agg(sol_first=("sol", "first"), sol_last=("sol", "last")).reset_index()
    g["cagr_pct"] = [_cagr(a, b, end - start) for a, b in zip(g.sol_first, g.sol_last)]
    g = g.sort_values("cagr_pct", ascending=False)
    return {"dataset_id": ds, "period": f"{start}–{end}", "n_units": g.shape[0],
            "growth_by_unit": _records(g, 25), "notes": _base_notes() + [f"Source: {df['source'].iloc[0]}"]}


@tool("Time series", core=True)
def detect_anomalies(start_year: Optional[int] = None, end_year: Optional[int] = None,
                     z_threshold: float = 2.5, iso3: Optional[str] = None) -> dict:
    """Detects abnormal months (drops or spikes) in national brightness: deseasonalisation, rolling-median
    trend and robust z-score. Each anomaly is classified as 'likely real drop', 'increase' or
    'likely artefact (clouds)' depending on cloud-free coverage. Useful for spotting power outages, crises,
    conflicts. Returns a dataset_id for plot_timeseries.

    Args:
        start_year: First year (default: 4 years before the latest complete monthly year).
        end_year: Last year (default: latest complete monthly year).
        z_threshold: Robust z-score threshold (2 = sensitive, 3 = strict).
        iso3: ISO3 country code. Default: the session country.
    """
    iso = _iso(iso3)
    end = int(end_year) if end_year else max(backend.monthly_full_years)
    start = int(start_year) if start_year else end - 3
    df = backend.national_series(iso, start, end, "monthly").copy()
    s = df.sol.astype(float)
    df["month"] = df.period.str[5:7].astype(int)
    if len(df) >= 24:
        ratio = s / s.rolling(12, center=True, min_periods=6).median()
        seas = ratio.groupby(df.month).median()
        df["seasonal_index"] = df.month.map(seas / seas.mean())
    else:
        df["seasonal_index"] = 1.0
    deseason = s / df.seasonal_index
    baseline = deseason.rolling(7, center=True, min_periods=3).median()
    resid = np.log(deseason / baseline)
    mad = np.median(np.abs(resid - np.median(resid.dropna()))) or 1e-9
    df["z"] = (resid - np.median(resid.dropna())) / (1.4826 * mad)
    df["is_anomaly"] = df.z.abs() >= z_threshold
    def label(r):
        if not r.is_anomaly:
            return ""
        if r.coverage_pct < 70:
            return "likely artefact (clouds)"
        return "likely real drop" if r.z < 0 else "increase"
    df["anomaly_type"] = df.apply(label, axis=1)
    ds = S.add_dataset(df, "anomalies", f"Monthly anomalies — {iso}", iso3=iso, frequency="monthly")
    an = df[df.is_anomaly]
    S.add_kpi("Months analysed", len(df))
    S.add_kpi("Anomalies detected", len(an))
    return {"dataset_id": ds, "period": f"{df.period.iloc[0]} → {df.period.iloc[-1]}", "z_threshold": z_threshold,
            "anomalies": _records(an, 20, ["period", "sol", "z", "coverage_pct", "anomaly_type"]),
            "notes": _base_notes() + ["An anomaly with coverage < 70% is more likely due to clouds.",
                                      "Confirm a real drop with an independent source (press, electricity utility)."]}

### 6.3 · Subnational spatial analysis

Beyond the ranking, `get_admin_stats` measures the **spatial concentration** of light (Gini index across units, share of the top unit) and `compare_periods` tests for **convergence**: are the least-lit regions catching up with the others?

In [ ]:
@tool("Spatial analysis", core=True)
def get_admin_stats(year: Optional[int] = None, level: int = 1,
                    sort_by: Literal["sol", "mean_rad", "lit_km2", "lit_share", "sol_share", "sol_density"] = "sol",
                    top_n: int = 10, ascending: bool = False, iso3: Optional[str] = None) -> dict:
    """Night-light statistics PER ADMINISTRATIVE UNIT for one year: SOL, mean radiance, lit area, lit share,
    share of national SOL, density (SOL/km²), with a ranking and spatial concentration measures
    (Gini, share of the top unit). Returns a dataset_id for plot_map (map) or plot_ranking (bar chart).

    Args:
        year: Year (default: latest available year).
        level: Administrative level: 1 = regions, 2 = districts.
        sort_by: Indicator used for the ranking.
        top_n: Number of units returned in the summary.
        ascending: True to rank from least to most lit.
        iso3: ISO3 country code. Default: the session country.
    """
    iso, level = _iso(iso3), _level(level)
    year = int(year or backend.default_year())
    df = backend.zonal_stats(iso, level, [year])
    df["sol_share"] = 100 * df.sol / df.sol.sum()
    df["sol_density"] = df.sol / df.area_km2
    df = df.sort_values(sort_by, ascending=ascending).reset_index(drop=True)
    df["rank"] = np.arange(1, len(df) + 1)
    ds = S.add_dataset(df, "admin_stats", f"ADM{level} statistics {year} — {iso}", iso3=iso, level=level, year=year)
    by_sol = df.sort_values("sol", ascending=False)
    conc = {"gini_sol_between_units": _gini(df.sol), "top1_unit": by_sol.unit.iloc[0],
            "top1_share_pct": by_sol.sol_share.iloc[0], "top3_share_pct": by_sol.sol_share.head(3).sum(),
            "n_units": len(df), "national_lit_share_pct": 100 * df.lit_km2.sum() / df.area_km2.sum()}
    S.add_kpi("Brightest unit", by_sol.unit.iloc[0], f"{by_sol.sol_share.iloc[0]:.0f}% of SOL")
    S.add_kpi("Spatial Gini of SOL", f"{conc['gini_sol_between_units']:.2f}")
    return {"dataset_id": ds, "year": year, "level": level, "sorted_by": sort_by, "concentration": conc,
            "top": _records(df, top_n, ["rank", "unit", "sol", "mean_rad", "lit_km2", "lit_share", "sol_share", "sol_density"]),
            "notes": _base_notes() + [f"Source: {df['source'].iloc[0]}",
                                      "Gini close to 1 = light concentrated in a few units."]}


@tool("Spatial analysis", core=True)
def compare_periods(year_start: int, year_end: int, level: int = 1, iso3: Optional[str] = None) -> dict:
    """Compares two years by administrative unit: absolute and % change in SOL, CAGR, change in lit area;
    identifies the largest increases and decreases and tests for convergence (are the least-lit units
    growing faster?). Returns a dataset_id for plot_map or plot_ranking (metric='pct_change').

    Args:
        year_start: Start year (e.g. 2015).
        year_end: End year (e.g. 2023).
        level: Administrative level: 1 = regions, 2 = districts.
        iso3: ISO3 country code. Default: the session country.
    """
    from scipy import stats
    iso, level = _iso(iso3), _level(level)
    y0, y1 = sorted((int(year_start), int(year_end)))
    df = backend.zonal_stats(iso, level, [y0, y1])
    a = df[df.year == y0].set_index("unit_id")
    b = df[df.year == y1].set_index("unit_id")
    out = pd.DataFrame({"unit": a.unit, "sol_start": a.sol, "sol_end": b.sol,
                        "lit_km2_start": a.lit_km2, "lit_km2_end": b.lit_km2}).reset_index()
    out["abs_change"] = out.sol_end - out.sol_start
    out["pct_change"] = 100 * (out.sol_end / out.sol_start.replace(0, np.nan) - 1)
    out["cagr_pct"] = [_cagr(s, e, y1 - y0) for s, e in zip(out.sol_start, out.sol_end)]
    out["lit_km2_change"] = out.lit_km2_end - out.lit_km2_start
    out = out.sort_values("pct_change", ascending=False).reset_index(drop=True)
    ds = S.add_dataset(out, "change", f"Change {y0}→{y1} ADM{level} — {iso}", iso3=iso, level=level,
                       year_start=y0, year_end=y1)
    nat = 100 * (out.sol_end.sum() / out.sol_start.sum() - 1)
    valid = out.dropna(subset=["cagr_pct"])
    conv = None
    if len(valid) >= 5:
        rho, p = stats.spearmanr(np.log(valid.sol_start + 1e-9), valid.cagr_pct)
        conv = {"spearman_initial_level_vs_growth": rho, "p_value": p,
                "reading": "convergence (the least-lit units are catching up)" if rho < -0.3 and p < 0.1 else
                           "divergence (the brightest units are widening the gap)" if rho > 0.3 and p < 0.1 else
                           "no clear relationship"}
    S.add_kpi(f"National change {y0}→{y1}", f"{nat:+.1f}", "%")
    S.add_kpi("Largest increase", out.unit.iloc[0], f"{out['pct_change'].iloc[0]:+.0f}%")
    return {"dataset_id": ds, "period": f"{y0}→{y1}", "national_sol_change_pct": nat,
            "top_gainers": _records(out, 5, ["unit", "sol_start", "sol_end", "pct_change", "cagr_pct"]),
            "top_decliners": _records(out.iloc[::-1], 5, ["unit", "sol_start", "sol_end", "pct_change", "cagr_pct"]),
            "convergence_test": conv,
            "notes": _base_notes() + ["Small, dimly lit units have very volatile % changes."]}

### 6.4 · Population, development and validation

Day 4 of the workshop ends with a decisive question: **is the NTL proxy fit for dissemination in your country, or does it remain a diagnostic tool?** These tools answer it with quantified evidence: population living in darkness (WorldPop), correlation with official World Bank indicators, and comparison with peer countries.

In [ ]:
WB_ALIASES = {
    "gdp": ("NY.GDP.MKTP.KD", "GDP (constant 2015 USD)"),
    "gdp_per_capita": ("NY.GDP.PCAP.KD", "GDP per capita (constant 2015 USD)"),
    "electricity_access": ("EG.ELC.ACCS.ZS", "Access to electricity (% of population)"),
    "electricity_access_rural": ("EG.ELC.ACCS.RU.ZS", "Access to electricity, rural (%)"),
    "population": ("SP.POP.TOTL", "Total population"),
    "urban_share": ("SP.URB.TOTL.IN.ZS", "Urban population (%)"),
    "electricity_consumption": ("EG.USE.ELEC.KH.PC", "Electricity consumption (kWh per capita)"),
}


def _worldbank(iso, code, start, end):
    def compute():
        url = f"https://api.worldbank.org/v2/country/{iso}/indicator/{code}?format=json&per_page=200&date={start}:{end}"
        js = requests.get(url, timeout=30).json()
        if len(js) < 2 or not js[1]:
            raise ValueError(f"No World Bank data for {code} / {iso}.")
        rows = [(int(r["date"]), r["value"]) for r in js[1] if r["value"] is not None]
        return pd.DataFrame(rows, columns=["year", "value"]).sort_values("year").reset_index(drop=True)
    return CACHE.get(["wb", iso, code, start, end, dt.date.today().isoformat()[:7]], compute).copy()


@tool("Population & validation")
def get_population_light_stats(year: Optional[int] = None, level: int = 1, iso3: Optional[str] = None) -> dict:
    """Combines night lights and population (WorldPop Global2, 100 m) by administrative unit: population, population
    living in UNLIT pixels (a proxy for the electricity access gap), and SOL per 1,000 inhabitants.
    Returns a dataset_id for plot_map / plot_ranking (metric='pop_unlit_share' or 'sol_per_1000').

    Args:
        year: NTL year (default: latest available). WorldPop Global2 population of the same year (2015–2030).
        level: Administrative level: 1 = regions, 2 = districts.
        iso3: ISO3 country code. Default: the session country.
    """
    iso, level = _iso(iso3), _level(level)
    year = int(year or backend.default_year())
    pop = backend.population_stats(iso, level, year)
    z = backend.zonal_stats(iso, level, [year])[["unit_id", "sol", "lit_share"]]
    df = pop.merge(z, on="unit_id", how="left")
    df["pop_unlit_share"] = 100 * df.pop_unlit / df["pop"].replace(0, np.nan)
    df["sol_per_1000"] = 1000 * df.sol / df["pop"].replace(0, np.nan)
    df = df.sort_values("pop_unlit_share", ascending=False).reset_index(drop=True)
    ds = S.add_dataset(df, "population_light", f"Population and light ADM{level} {year} — {iso}",
                       iso3=iso, level=level, year=year)
    nat_share = 100 * df.pop_unlit.sum() / df["pop"].sum()
    S.add_kpi("Population in unlit areas", f"{nat_share:.0f}", "%")
    S.add_kpi("Population (WorldPop)", fmt(df["pop"].sum()), f"in {int(df.pop_year.iloc[0])}")
    notes = _base_notes() + [
        "An unlit pixel does not mean no electricity (low consumption, off-grid solar, "
        "lights switched off at night): it is a proxy, to be compared with the official access rate (get_worldbank_indicator)."]
    src_pop, py = df.pop_source.iloc[0], int(df.pop_year.iloc[0])
    notes.append(f"Population: {src_pop}, year {py}.")
    if py != year:
        notes.append(f"{py} population applied to {year} lights (closest available year).")
    if "projection" in src_pop:
        notes.append("Projected population (after 2025): not an observed estimate.")
    return {"dataset_id": ds, "year": year, "population_year": py, "population_source": src_pop,
            "national": {"population": df["pop"].sum(), "pop_unlit": df.pop_unlit.sum(), "pop_unlit_share_pct": nat_share},
            "by_unit": _records(df, 15, ["unit", "pop", "pop_unlit_share", "sol_per_1000", "lit_share"]), "notes": notes}


@tool("Population & validation")
def get_worldbank_indicator(indicator: str = "gdp", start_year: Optional[int] = None,
                            end_year: Optional[int] = None, iso3: Optional[str] = None) -> dict:
    """Retrieves an official World Bank indicator (WDI API). Accepted aliases: gdp, gdp_per_capita,
    electricity_access, electricity_access_rural, population, urban_share, electricity_consumption — or a raw
    WDI code (e.g. 'EG.ELC.ACCS.ZS').

    Args:
        indicator: Alias or WDI code of the indicator.
        start_year: First year (default: 2012).
        end_year: Last year (default: current year).
        iso3: ISO3 country code. Default: the session country.
    """
    iso = _iso(iso3)
    code, label = WB_ALIASES.get(indicator, (indicator, indicator))
    start, end = int(start_year or 2012), int(end_year or dt.date.today().year)
    df = _worldbank(iso, code, start, end)
    ds = S.add_dataset(df.rename(columns={"value": "indicator"}).assign(period=lambda d: d.year.astype(str)),
                       "worldbank", f"{label} — {iso}", iso3=iso, indicator=label)
    return {"dataset_id": ds, "indicator": label, "code": code, "data": _records(df, 40),
            "source": "World Bank, World Development Indicators (API v2)"}


@tool("Population & validation")
def correlate_ntl_with_indicator(indicator: str = "gdp", start_year: Optional[int] = None,
                                 end_year: Optional[int] = None, iso3: Optional[str] = None) -> dict:
    """VALIDATION of the proxy: correlates the national annual SOL series with an official World Bank
    indicator (aliases: gdp, gdp_per_capita, electricity_access, population, urban_share…). Computes Pearson's r,
    Spearman's rho and the log-log elasticity (% change in the indicator for +1% SOL). Returns a dataset_id
    for plot_scatter (x='sol', y='indicator').

    Args:
        indicator: Alias or WDI code of the indicator.
        start_year: First year (default: first available NTL year).
        end_year: Last year (default: latest available NTL year).
        iso3: ISO3 country code. Default: the session country.
    """
    from scipy import stats
    iso = _iso(iso3)
    start, end = _years(start_year, end_year, default_span=15)
    code, label = WB_ALIASES.get(indicator, (indicator, indicator))
    ntl = backend.national_series(iso, start, end, "annual")[["period", "sol"]]
    ntl["year"] = ntl.period.astype(int)
    wb = _worldbank(iso, code, start, end).rename(columns={"value": "indicator"})
    df = ntl.merge(wb, on="year").dropna()
    if len(df) < 4:
        raise ValueError(f"Only {len(df)} overlapping years: correlation not possible. Widen the period.")
    pr = stats.pearsonr(df.sol, df.indicator)
    sr = stats.spearmanr(df.sol, df.indicator)
    el = stats.linregress(np.log(df.sol), np.log(df.indicator)) if (df.indicator > 0).all() else None
    ds = S.add_dataset(df, "correlation", f"SOL vs {label} — {iso}", iso3=iso, indicator=label)
    S.add_kpi("Correlation r (Pearson)", f"{pr[0]:.2f}")
    if el:
        S.add_kpi("Log-log elasticity", f"{el.slope:.2f}")
    return {"dataset_id": ds, "indicator": label, "n_years": len(df), "years": f"{df.year.min()}–{df.year.max()}",
            "pearson_r": pr[0], "pearson_p": pr[1], "spearman_rho": sr[0],
            "elasticity_loglog": el.slope if el else None, "elasticity_r2": el.rvalue ** 2 if el else None,
            "notes": _base_notes() + [
                "Two trending series are often correlated without any causal link (spurious correlation).",
                "With fewer than 10 points, uncertainty is very high: draw conclusions with caution.",
                "The most convincing validation is subnational (NTL vs official regional statistics)."]}


@tool("Population & validation")
def compare_countries(iso3_list: str, year: Optional[int] = None) -> dict:
    """Compares several countries for one year: SOL, lit area, mean radiance, SOL/km², SOL per 1,000
    inhabitants and access to electricity (World Bank). Ideal for benchmarking against peer countries.
    Returns a dataset_id for plot_ranking.

    Args:
        iso3_list: Comma-separated ISO3 codes, e.g. 'RWA,BDI,UGA,KEN'.
        year: Year (default: latest available year).
    """
    year = int(year or backend.default_year())
    rows = []
    for code in [c.strip().upper() for c in iso3_list.split(",") if c.strip()][:8]:
        iso = _iso(code)
        nat = backend.national_series(iso, year, year, "annual").iloc[0]
        area = float(backend.admin_units(iso, 0).area_km2.sum())
        r = dict(iso3=iso, unit=country_name(iso), sol=nat.sol, mean_rad=nat.mean_rad, lit_km2=nat.lit_km2,
                 lit_share=nat.lit_share, sol_density=nat.sol / area)
        for key, col in (("population", "population"), ("electricity_access", "electricity_access")):
            try:
                w = _worldbank(iso, WB_ALIASES[key][0], year - 3, year)
                r[col] = w.value.iloc[-1]
            except Exception:
                r[col] = None
        if r["population"]:
            r["sol_per_capita_1000"] = 1000 * r["sol"] / r["population"]
        rows.append(r)
    df = pd.DataFrame(rows).sort_values("sol", ascending=False).reset_index(drop=True)
    ds = S.add_dataset(df, "countries", f"Country comparison {year}", year=year)
    return {"dataset_id": ds, "year": year, "comparison": _records(df, 10),
            "notes": _base_notes() + ["Preferably compare relative indicators (per km², per capita)."]}

### 6.5 · Events and quality control

In [ ]:
@tool("Events & quality")
def assess_event_impact(event_date: str, days_before: int = 30, days_after: int = 30, level: int = 1,
                        iso3: Optional[str] = None) -> dict:
    """Measures the impact of a dated event (power outage, flood, conflict, cyclone, festival…) using
    NASA Black Marble DAILY images: median of good-quality nights before vs after, by administrative
    unit. Returns a dataset_id for plot_map / plot_ranking (metric='pct_change').

    Args:
        event_date: Event date in YYYY-MM-DD format.
        days_before: Number of days in the window before the event.
        days_after: Number of days in the window after the event.
        level: Administrative level: 1 = regions, 2 = districts.
        iso3: ISO3 country code. Default: the session country.
    """
    iso, level = _iso(iso3), _level(level)
    pd.Timestamp(event_date)                       # validates the format
    df = backend.event_impact(iso, level, event_date, int(days_before), int(days_after))
    df["pct_change"] = 100 * (df.sol_after / df.sol_before.replace(0, np.nan) - 1)
    df = df.sort_values("pct_change").reset_index(drop=True)
    ds = S.add_dataset(df, "event", f"Impact of the event on {event_date} — {iso}", iso3=iso, level=level)
    nat = 100 * (df.sol_after.sum() / df.sol_before.sum() - 1)
    S.add_kpi("National change after the event", f"{nat:+.1f}", "%")
    S.add_kpi("Most affected unit", df.unit.iloc[0], f"{df['pct_change'].iloc[0]:+.0f}%")
    return {"dataset_id": ds, "event_date": event_date, "window_days": [days_before, days_after],
            "national_change_pct": nat, "images_before": int(df.n_images_before.iloc[0]),
            "images_after": int(df.n_images_after.iloc[0]),
            "most_affected": _records(df, 8, ["unit", "sol_before", "sol_after", "pct_change", "coverage_after_pct"]),
            "notes": _base_notes() + ["Changes of ±10% are common even without an event (moon, clouds, viewing angle).",
                                      "Compare with the same window in the previous year to isolate the effect."]}


GAS_FLARING = {"NGA", "DZA", "LBY", "EGY", "AGO", "GAB", "COG", "GNQ", "CMR", "TCD", "SDN", "SSD", "TUN", "CIV", "GHA", "COD"}


@tool("Events & quality", core=False)
def get_quality_report(year: Optional[int] = None, iso3: Optional[str] = None) -> dict:
    """NTL data QUALITY report for one year: cloud-free coverage month by month, doubtful months,
    radiance distribution (median, p90, p99, maximum) and likely artefacts (gas flares, saturation, clouds).
    Use it before interpreting a drop or a surprising value.

    Args:
        year: Year (default: latest available year).
        iso3: ISO3 country code. Default: the session country.
    """
    iso = _iso(iso3)
    year = int(year or backend.default_year())
    q = backend.quality_report(iso, year)
    cov = q["monthly_coverage"]
    ds = S.add_dataset(cov, "quality", f"Monthly coverage {year} — {iso}", iso3=iso, frequency="monthly")
    p = q["percentiles"]
    flags = []
    bad = cov[cov.coverage_pct < 70]
    if len(bad):
        flags.append(f"Clouds: {len(bad)} months with coverage < 70% ({', '.join(bad.period)}).")
    if p.get("max") and p.get("p99") and p["max"] > 5 * p["p99"]:
        flags.append("Isolated extreme values (max ≫ p99): gas flares, industrial sites or saturated pixels possible.")
    if iso in GAS_FLARING:
        flags.append("Country with known gas flaring: check very bright pixels outside cities.")
    S.add_kpi("Mean coverage", f"{cov.coverage_pct.mean():.0f}", "%")
    return {"dataset_id": ds, "year": year, "radiance_percentiles_lit_pixels": p,
            "monthly_coverage": _records(cov, 12), "flags": flags or ["No major artefact detected automatically."],
            "notes": _base_notes() + ["Blooming (urban halo) and rural noise are not detectable by this simple test."]}

### 6.6 · Visualisation and export

The charting tools receive **no data** from the LLM: only a `dataset_id`. The LLM therefore cannot “invent” the plotted values. Charts are interactive (hover, zoom), saved as HTML in `ntl_agent_outputs/figures/` and included in the final report.

In [ ]:
PCT_METRICS = {"pct_change", "cagr_pct", "lit_share", "sol_share", "pop_unlit_share", "coverage_pct",
               "electricity_access", "abs_change", "lit_km2_change"}
DIVERGING = {"pct_change", "cagr_pct", "abs_change", "lit_km2_change", "z"}
WARM = ["#FBEFD9", "#F5C242", "#D49A00", "#C4621D", "#B83B2E"]   # “more = less favourable”
NEGATIVE_METRICS = {"pop_unlit_share", "pop_unlit"}


def _style(fig, title, ytitle=None, xtitle=None, height=470):
    src = f"Source: {backend.name}"
    if not backend.synthetic:
        src = "Sources: NOAA/EOG VIIRS, NASA Black Marble, WorldPop, geoBoundaries, World Bank — via Earth Engine"
    src += f" · lit threshold {LIT_THRESHOLD} nW/cm²/sr · NTL Agent STG17"
    fig.update_layout(title=dict(text=f"<b>{title}</b>"), height=height,
                      legend=dict(orientation="h", yanchor="bottom", y=1.01, xanchor="right", x=1))
    if ytitle is not None:
        fig.update_yaxes(title_text=ytitle)
    if xtitle is not None:
        fig.update_xaxes(title_text=xtitle)
    fig.add_annotation(text=f"<i>{src}</i>", xref="paper", yref="paper", x=0, y=-0.13, xanchor="left",
                       yanchor="top", showarrow=False, font=dict(size=10, color=C["slate"]))
    if backend.synthetic:
        fig.add_annotation(text="DEMO — SYNTHETIC DATA", xref="paper", yref="paper", x=0.5, y=0.5,
                           showarrow=False, textangle=-18, font=dict(size=30, color="rgba(184,59,46,0.16)"))
    return fig


def _require(df, metric):
    if metric not in df.columns:
        num = [c for c in df.select_dtypes("number").columns if c not in ("n_px", "year", "month", "rank")]
        raise ValueError(f"Column “{metric}” is not in this dataset. Possible numeric columns: {num}")


def _polygons(g):
    """Extract only the polygons from a geometry. After simplification, Earth Engine may return
    GeometryCollections mixing polygons and segments (LineString): those segments have no area to fill."""
    from shapely.geometry import Polygon, MultiPolygon, GeometryCollection
    if g is None or g.is_empty:
        return []
    if isinstance(g, Polygon):
        return [g]
    if isinstance(g, (MultiPolygon, GeometryCollection)):
        return [p for part in g.geoms for p in _polygons(part)]
    return []                                   # LineString, Point… ignored


def _one_year(df, year):
    if "year" in df.columns and df.year.nunique() > 1:
        return df[df.year == (int(year) if year else df.year.max())]
    return df


@tool("Visualisation", core=True)
def plot_timeseries(dataset_id: str, metric: str = "sol", title: Optional[str] = None, top_n: int = 8) -> dict:
    """Plot how an indicator changes over time from a dataset_id (annual or monthly national series,
    anomalies, panel of regions). Adds the trend line, anomalies and cloud-free coverage when they
    are available.

    Args:
        dataset_id: Identifier returned by a data tool (e.g. 'ds02_national_annual').
        metric: Column to plot: sol, mean_rad, lit_km2, lit_share, coverage_pct…
        title: Chart title (short, informative).
        top_n: For a panel of regions: number of regions plotted (the highest in the latest year).
    """
    ds = S.get(dataset_id)
    df = ds["df"].copy()
    _require(df, metric)
    title = title or ds["title"]
    fig = go.Figure()
    if "unit" in df.columns and "year" in df.columns and df.unit.nunique() > 1:
        last = df[df.year == df.year.max()].nlargest(top_n, metric).unit.tolist()
        for i, u in enumerate(last):
            sub = df[df.unit == u].sort_values("year")
            fig.add_scatter(x=sub.year, y=sub[metric], mode="lines+markers", name=u,
                            line=dict(width=3 if i == 0 else 2, color=SERIES[i % len(SERIES)]), marker=dict(size=6))
        xt = "Year"
    else:
        xcol = "period" if "period" in df.columns else "year"
        d = df.sort_values(xcol)
        monthly = d[xcol].astype(str).str.len().eq(7).all()
        xs = pd.to_datetime(d[xcol].astype(str)) if monthly else d[xcol].astype(int)
        if not monthly:
            fig.update_xaxes(dtick=1, tickformat="d")
        if monthly and "coverage_pct" in d.columns and metric != "coverage_pct":
            fig.add_bar(x=xs, y=d.coverage_pct, name="Cloud-free coverage (%)", yaxis="y2",
                        marker_color="rgba(169,181,176,0.35)", hovertemplate="%{y:.0f}%<extra>coverage</extra>")
            fig.update_layout(yaxis2=dict(overlaying="y", side="right", range=[0, 300], showgrid=False,
                                          showticklabels=False, title=None))
        fig.add_scatter(x=xs, y=d[metric], mode="lines+markers", name=LABELS.get(metric, metric),
                        line=dict(color=C["green"], width=3), marker=dict(size=5 if monthly else 8, color=C["deep"]),
                        fill="tozeroy", fillcolor="rgba(0,168,106,0.07)")
        if not monthly and len(d) >= 4:
            xn = np.arange(len(d))
            coef = np.polyfit(xn, d[metric].astype(float), 1)
            fig.add_scatter(x=xs, y=np.polyval(coef, xn), mode="lines", name="Linear trend",
                            line=dict(color=C["ochre"], width=2, dash="dash"))
        if "is_anomaly" in d.columns:
            colors = {"likely real drop": C["brick"], "likely artefact (clouds)": C["ochre"], "increase": C["teal"]}
            for lab, col in colors.items():
                a = d[d.anomaly_type == lab]
                if len(a):
                    fig.add_scatter(x=pd.to_datetime(a.period), y=a[metric], mode="markers", name=lab,
                                    marker=dict(size=15, color=col, symbol="circle-open", line=dict(width=3)))
        fig.add_annotation(x=xs.iloc[-1], y=d[metric].iloc[-1], text=f"<b>{fmt(d[metric].iloc[-1])}</b>",
                           showarrow=False, yshift=16, font=dict(color=C["deep"]))
        xt = "Month" if monthly else "Year"
    _style(fig, title, LABELS.get(metric, metric), xt)
    fig_id = S.add_figure(fig, title)
    return {"figure_id": fig_id, "title": title, "message": "Chart created; it will be displayed below the answer."}


@tool("Visualisation", core=True)
def plot_ranking(dataset_id: str, metric: str = "sol", top_n: int = 15, title: Optional[str] = None,
                 year: Optional[int] = None) -> dict:
    """Horizontal bar chart ranking units (regions, districts or countries) by an indicator.
    Diverging colours (green/red) for changes.

    Args:
        dataset_id: Identifier of a per-unit dataset (get_admin_stats, compare_periods, get_population_light_stats…).
        metric: Column to rank by: sol, mean_rad, lit_share, sol_share, pct_change, pop_unlit_share…
        top_n: Number of bars.
        title: Chart title.
        year: For a multi-year panel: year to show (default: the most recent).
    """
    ds = S.get(dataset_id)
    df = _one_year(ds["df"].copy(), year)
    _require(df, metric)
    title = title or f"{LABELS.get(metric, metric)} — {ds['title']}"
    d = df.dropna(subset=[metric]).sort_values(metric, ascending=False)
    div = metric in DIVERGING or (d[metric] < 0).any()
    if len(d) > top_n:
        d = pd.concat([d.head(top_n - top_n // 3), d.tail(top_n // 3)]) if div else d.head(top_n)
    d = d.iloc[::-1]
    if div:
        colors = [C["green"] if v >= 0 else C["brick"] for v in d[metric]]
    else:
        ramp = WARM if metric in NEGATIVE_METRICS else RAMP
        idx = np.linspace(1, len(ramp) - 1, len(d)).round().astype(int)
        colors = [ramp[i] for i in idx]
    suffix = " %" if metric in PCT_METRICS and metric not in ("abs_change", "lit_km2_change") else ""
    fig = go.Figure(go.Bar(x=d[metric], y=d.unit, orientation="h", marker_color=colors,
                           text=[f"{fmt(v)}{suffix}" for v in d[metric]], textposition="outside", cliponaxis=False,
                           hovertemplate="%{y}<br>%{x:,.2f}<extra></extra>"))
    fig.update_xaxes(showticklabels=False, showgrid=False)
    fig.update_yaxes(gridcolor="rgba(0,0,0,0)")
    _style(fig, title, None, LABELS.get(metric, metric), height=max(380, 30 * len(d) + 160))
    fig_id = S.add_figure(fig, title)
    return {"figure_id": fig_id, "title": title, "message": "Chart created; it will be displayed below the answer."}


@tool("Visualisation", core=True)
def plot_map(dataset_id: str, metric: str = "sol", title: Optional[str] = None, year: Optional[int] = None) -> dict:
    """Choropleth map of administrative units coloured by an indicator (automatic logarithmic scale for
    highly skewed indicators such as SOL; diverging scale for changes).

    Args:
        dataset_id: Identifier of a per-administrative-unit dataset (get_admin_stats, compare_periods,
            get_population_light_stats, assess_event_impact, get_admin_timeseries).
        metric: Column to map: sol, mean_rad, lit_share, sol_density, pct_change, pop_unlit_share…
        title: Map title.
        year: For a multi-year panel: year to map (default: the most recent).
    """
    ds = S.get(dataset_id)
    meta = ds["meta"]
    if "level" not in meta or "unit_id" not in ds["df"].columns:
        raise ValueError("This dataset is not broken down by administrative unit: use plot_ranking or plot_timeseries.")
    df = _one_year(ds["df"].copy(), year)
    _require(df, metric)
    gj = backend.geojson(meta["iso3"], meta["level"])
    title = title or f"{LABELS.get(metric, metric)} — {ds['title']}"
    v = df[metric].astype(float)
    div = metric in DIVERGING or (v < 0).any()
    cb = dict(title=dict(text=LABELS.get(metric, metric).replace(" (", "<br>("), side="top"), thickness=14, len=0.75)
    if div:
        lim = float(np.nanmax(np.abs(v))) or 1
        df["_c"] = v
        kw = dict(color_continuous_scale=[[0, C["brick"]], [0.5, "#F4F1EA"], [1, C["green"]]], range_color=(-lim, lim))
    elif v.min() > 0 and v.max() / v.min() > 50:
        df["_c"] = np.log10(v)
        ticks = np.arange(math.floor(df._c.min()), math.ceil(df._c.max()) + 1)
        cb.update(tickvals=ticks, ticktext=[fmt(10.0 ** float(t)) for t in ticks])
        kw = dict(color_continuous_scale=RAMP)
    else:
        df["_c"] = v
        kw = dict(color_continuous_scale=WARM if metric in NEGATIVE_METRICS else RAMP)
    # Map drawn as "Cartesian" Plotly polygons: no basemap to download,
    # so it also works offline (Kaggle without Internet, filtered corporate network…).
    from plotly.colors import sample_colorscale
    scale = kw["color_continuous_scale"]
    lo, hi = kw.get("range_color", (float(np.nanmin(df._c)), float(np.nanmax(df._c))))
    span = (hi - lo) or 1.0
    vals = dict(zip(df.unit_id, zip(df._c, v, df.unit)))
    fig = go.Figure()
    lats = []
    for f in gj["features"]:
        uid = f["properties"]["unit_id"]
        g = shape(f["geometry"])
        polys = _polygons(g)
        c, raw, name = vals.get(uid, (np.nan, np.nan, f["properties"].get("unit", uid)))
        col = "#E6EBE8" if pd.isna(c) else sample_colorscale(scale, [min(max((c - lo) / span, 0), 1)])[0]
        tip = f"<b>{name}</b><br>{LABELS.get(metric, metric)}: {fmt(raw)}"
        for i, poly in enumerate(polys):
            xs, ys = poly.exterior.xy
            lats += list(ys)
            fig.add_scatter(x=list(xs), y=list(ys), fill="toself", fillcolor=col, mode="lines",
                            line=dict(color="white", width=0.8), hoveron="fills", text=tip, hoverinfo="text",
                            showlegend=False, legendgroup=uid)
    fig.add_scatter(x=[None], y=[None], mode="markers", showlegend=False, hoverinfo="skip",
                    marker=dict(colorscale=scale, cmin=lo, cmax=hi, color=[lo], showscale=True, colorbar=cb))
    if len(df) <= 35:                               # labels at the centre of each unit
        pts = []
        for f in gj["features"]:
            ps = _polygons(shape(f["geometry"]))
            if ps:
                pts.append((max(ps, key=lambda q: q.area).representative_point(), f["properties"].get("unit", "")))
        fig.add_scatter(x=[p.x for p, _ in pts], y=[p.y for p, _ in pts], text=[n for _, n in pts], mode="text",
                        textfont=dict(size=10, color=C["ink"]), hoverinfo="skip", showlegend=False)
    lat_mid = float(np.mean(lats)) if lats else 0.0
    fig.update_xaxes(visible=False, showgrid=False)
    fig.update_yaxes(visible=False, showgrid=False, scaleanchor="x", scaleratio=1 / max(math.cos(math.radians(lat_mid)), 0.2))
    fig.update_layout(margin=dict(l=10, r=10, t=70, b=70), hovermode="closest")
    _style(fig, title, height=580)
    fig_id = S.add_figure(fig, title)
    return {"figure_id": fig_id, "title": title, "message": "Map created; it will be displayed below the answer."}


@tool("Visualisation")
def plot_scatter(dataset_id: str, x: str = "sol", y: str = "indicator", log_scale: bool = True,
                 title: Optional[str] = None) -> dict:
    """Scatter plot of two columns of a dataset (e.g. SOL vs GDP after correlate_ntl_with_indicator), with a
    fitted line and labels (years or units).

    Args:
        dataset_id: Dataset identifier.
        x: Column on the x-axis.
        y: Column on the y-axis.
        log_scale: Logarithmic axes (recommended for SOL and GDP).
        title: Chart title.
    """
    ds = S.get(dataset_id)
    df = _one_year(ds["df"].copy(), None).dropna(subset=[x, y])
    _require(df, x); _require(df, y)
    lab = df["year"].astype(str) if "year" in df.columns and df.year.nunique() > 1 else df.get("unit", df.index.astype(str))
    title = title or f"{LABELS.get(y, ds['meta'].get('indicator', y))} vs {LABELS.get(x, x)}"
    fig = go.Figure(go.Scatter(x=df[x], y=df[y], mode="markers+text", text=lab, textposition="top center",
                               marker=dict(size=11, color=C["green"], line=dict(color="white", width=1)),
                               textfont=dict(size=10, color=C["slate"]), name="observations"))
    xx, yy = df[x].astype(float), df[y].astype(float)
    ok = (xx > 0) & (yy > 0) if log_scale else pd.Series(True, index=df.index)
    if ok.sum() >= 3:
        fx, fy = (np.log(xx[ok]), np.log(yy[ok])) if log_scale else (xx[ok], yy[ok])
        coef = np.polyfit(fx, fy, 1)
        grid = np.linspace(fx.min(), fx.max(), 50)
        fig.add_scatter(x=np.exp(grid) if log_scale else grid,
                        y=np.exp(np.polyval(coef, grid)) if log_scale else np.polyval(coef, grid), mode="lines",
                        line=dict(color=C["ochre"], dash="dash"), name=f"fit (slope {coef[0]:.2f})")
    if log_scale:
        fig.update_xaxes(type="log")
        fig.update_yaxes(type="log")
    _style(fig, title, LABELS.get(y, ds["meta"].get("indicator", y)), LABELS.get(x, x))
    fig_id = S.add_figure(fig, title)
    return {"figure_id": fig_id, "title": title, "message": "Chart created; it will be displayed below the answer."}


@tool("Visualisation")
def show_night_image(year: Optional[int] = None, iso3: Optional[str] = None) -> dict:
    """Display the satellite image of the country's night-time lights for a given year (logarithmic scale,
    dark background). Ideal for a first visual impression or for spotting gas flares and urban halos.

    Args:
        year: Year (default: latest available).
        iso3: ISO3 country code. Default: the session country.
    """
    iso = _iso(iso3)
    year = int(year or backend.default_year())
    png = backend.thumbnail(iso, year)
    title = f"{country_name(iso)} at night — {year}"
    fig_id = S.add_figure(png, title, kind="image")
    return {"figure_id": fig_id, "title": title, "message": "Image created; it will be displayed below the answer."}


@tool("Export")
def export_dataset(dataset_id: str, file_format: Literal["csv", "xlsx"] = "csv") -> dict:
    """Export a session dataset as CSV or Excel to the outputs folder (for publication or reuse).

    Args:
        dataset_id: Identifier of the dataset to export.
        file_format: 'csv' or 'xlsx'.
    """
    ds = S.get(dataset_id)
    path = OUT / "data" / f"{dataset_id}.{file_format}"
    df = ds["df"].drop(columns=[c for c in ("n_px",) if c in ds["df"].columns])
    (df.to_csv(path, index=False) if file_format == "csv" else df.to_excel(path, index=False))
    return {"path": str(path), "rows": len(df), "columns": list(df.columns)}


@tool("Export")
def list_session_datasets() -> dict:
    """List the datasets already computed in this session (identifier, title, columns), so they can be reused
    in a chart or an export without recomputing."""
    return {"datasets": [{"dataset_id": k, "title": v["title"], "rows": len(v["df"]),
                          "columns": [c for c in v["df"].columns if c != "n_px"]} for k, v in S.datasets.items()][-15:]}

In [ ]:
def show_toolbox():
    fam_color = {"Discovery": C["green"], "Time series": C["deep"], "Spatial analysis": C["teal"],
                 "Population & validation": C["ochre"], "Events & quality": C["terra"],
                 "Visualisation": C["brick"], "Export": C["slate"], "Extension": C["forest"]}
    html = ""
    for fam in dict.fromkeys(t["family"] for t in TOOLS.values()):
        cards = ""
        for name, t in TOOLS.items():
            if t["family"] != fam:
                continue
            desc = t["schema"]["function"]["description"]
            short = desc.split(". ")[0][:150]
            core = ' <span class="ntl-pill" style="font-size:10px;padding:0 6px">core</span>' if t["core"] else ""
            cards += f'<div class="ntl-tool"><b>{name}</b>{core}<div class="ntl-muted">{short}</div></div>'
        col = fam_color.get(fam, C["green"])
        html += (f'<div style="margin:12px 0 6px;font-weight:700;color:{col};letter-spacing:2px;'
                 f'text-transform:uppercase;font-size:12px">● {fam}</div><div class="ntl-grid">{cards}</div>')
    n_core = sum(t["core"] for t in TOOLS.values())
    ui.card(f"{len(TOOLS)} tools available ({n_core} in the “core” set)", html, kicker="06 · Toolbox")

show_toolbox()

## 07 · Testing the tools by hand (without an LLM)

Before handing the tools over to an agent, **check them yourself**. This is also what the LLM “sees”: a compact JSON dictionary. If a human cannot understand a tool's output, neither can the LLM.

In [ ]:
S.new_turn()
profile = get_country_profile()
series = get_national_timeseries()
ui.card(f"{profile['country']} — first results", ui.kpis(S.kpis) +
        f'<div class="ntl-muted">Dataset created: <code>{series["dataset_id"]}</code> · {series["notes"][-1]}</div>',
        kicker="07 · Manual test")
fig_id = plot_timeseries(series["dataset_id"], "sol", f"Sum of radiances —{profile['country']}")["figure_id"]
S.figures[fig_id]["obj"].show()

In [ ]:
# Independent of execution order: recomputes whatever is missing.
if "profile" not in globals():
    S.new_turn()
    profile = get_country_profile()
stats = get_admin_stats()
S.figures[plot_map(stats["dataset_id"], "mean_rad")["figure_id"]]["obj"].show()
img = S.figures[show_night_image()["figure_id"]]["obj"]
ui.card(f"{profile['country']} at night",
        f'<div style="background:#000814;border-radius:10px;padding:12px;text-align:center">'
        f'<img src="data:image/png;base64,{base64.b64encode(img).decode()}" style="max-width:100%;max-height:520px"></div>',
        kicker="Satellite image")

### Anatomy of a tool call

On the left, what the LLM receives **before** acting: the JSON schema generated automatically from the function signature and docstring. On the right, what it receives **after** the call: a compact summary, caution notes and a `dataset_id`, but **not** the full table.

In [ ]:
import html as html_lib
# If the previous cell was not run, recompute what we need.
if "stats" not in globals():
    S.new_turn()
    stats = get_admin_stats()
schema = json.dumps(TOOLS["get_admin_stats"]["schema"], ensure_ascii=False, indent=2)
result = json.dumps(stats, ensure_ascii=False, indent=2, default=str)
ui.card("What the LLM sees",
        '<div style="display:grid;grid-template-columns:1fr 1fr;gap:12px">'
        f'<div><div class="ntl-kicker">① Tool schema (before)</div><pre style="max-height:420px">{html_lib.escape(schema)}</pre></div>'
        f'<div><div class="ntl-kicker">② Returned result (after)</div><pre style="max-height:420px">{html_lib.escape(result[:3500])}</pre></div>'
        '</div>', kicker="Anatomy of a call")

## 08 · The agent: the reason → act → observe loop

```
question ──► LLM ──► tool calls? ──yes──► Python execution ──► JSON results ──┐
               ▲                    │                                          │
               └────────────────────┼──────────────────────────────────────────┘
                                    no
                                    ▼
                       final answer (Markdown) + KPIs + charts + trace
```

Three elements determine the quality of an agent:

1. **The system prompt**: role, context (country, available data), rules (no figure without a tool, always state the limitations) and an **imposed answer format**. It is the agent's “job contract”.
2. **The loop**: up to `MAX_AGENT_STEPS` round trips. Tool errors are sent back to the LLM so it can correct itself; a malformed call is requested again.
3. **The output**: a formatted Markdown answer, key indicators, charts and a **complete trace** of the calls (tool, arguments, duration) for auditing. A statistical result must remain verifiable.

In [ ]:
import html as html_lib


def _clean(o):
    """Make a tool result serialisable and compact (rounding, NaN → null, numpy types → Python)."""
    if isinstance(o, dict):
        return {k: _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_clean(v) for v in o]
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (float, np.floating)):
        o = float(o)
        if math.isnan(o) or math.isinf(o):
            return None
        return round(o, 3) if abs(o) < 1000 else round(o, 1)
    if isinstance(o, (np.bool_,)):
        return bool(o)
    if isinstance(o, (pd.Timestamp, dt.date)):
        return str(o)
    return o


def _coerce_args(fn, args):
    """Tolerate minor LLM typing mistakes ('2020' instead of 2020, unknown arguments)."""
    params = inspect.signature(fn).parameters
    clean, dropped = {}, []
    for k, v in (args or {}).items():
        if k not in params:
            dropped.append(k)
            continue
        ann = params[k].annotation
        if get_origin(ann) is Union:
            ann = [a for a in get_args(ann) if a is not type(None)][0]
        if v is None or v == "":
            continue
        try:
            if ann is int:
                v = int(float(v))
            elif ann is float:
                v = float(v)
            elif ann is bool:
                v = v if isinstance(v, bool) else str(v).lower() in ("true", "1", "yes", "oui")
            elif ann is str or get_origin(ann) is Literal:
                v = str(v)
        except (TypeError, ValueError):
            pass
        clean[k] = v
    return clean, dropped


MAX_TOOL_CHARS = 7000


def execute_tool(name, args, allowed):
    t0 = time.time()
    if name not in allowed:
        out = {"ok": False, "error": f"Unknown tool: {name}", "available_tools": sorted(allowed)}
    elif args is None:
        out = {"ok": False, "error": "Unreadable JSON arguments.", "hint": "Send back a valid JSON object."}
    else:
        fn = TOOLS[name]["fn"]
        clean, dropped = _coerce_args(fn, args)
        try:
            out = {"ok": True, **fn(**clean)}
            if dropped:
                out["ignored_arguments"] = dropped
        except Exception as e:
            out = {"ok": False, "error": f"{type(e).__name__}: {str(e)[:500]}",
                   "hint": "Fix the arguments (available years, level 1 or 2, exact region name, "
                           "existing dataset_id) and retry, or explain the limitation to the user."}
    txt = json.dumps(_clean(out), ensure_ascii=False, default=str)
    if len(txt) > MAX_TOOL_CHARS:
        txt = txt[:MAX_TOOL_CHARS] + ' …[truncated: use the dataset_id for follow-up steps]"}'
    short = out.get("dataset_id") or out.get("figure_id") or out.get("path") or out.get("error", "")
    rec = dict(tool=name, args=args, ok=bool(out.get("ok")), duration=time.time() - t0, result=str(short)[:160])
    return txt, rec


def build_system_prompt(iso3=None):
    iso = _iso(iso3)
    synth = ("\n⚠️ The data are SYNTHETIC (demo mode): start your answer with an explicit warning "
             "and never present these figures as real.") if backend.synthetic else ""
    return f"""You are “Lumina”, a senior geospatial analyst specialising in night-time lights (VIIRS), working for
the national statistics office of {country_name(iso)} ({iso}), as part of the STG17 workshop (AfDB / AU STATAFRIC).

CONTEXT
- Default country: {country_name(iso)} (ISO3 {iso}). You may analyse other countries if asked (iso3 parameter).
- Data: {backend.name}. Annual: {min(backend.annual_years)}–{max(backend.annual_years)}; monthly:
  {backend.availability['monthly_months'][0]} → {backend.availability['monthly_months'][-1]}.
- “Lit” pixel: radiance > {LIT_THRESHOLD} nW/cm²/sr. SOL = sum of radiances over an area.
- Today's date: {dt.date.today().isoformat()}.{synth}

RULES
1. No figure without a tool: every value you cite comes from a tool result in this conversation.
2. Plan: first the data tools, then the charts (plot_*, show_night_image) using the dataset_id
   obtained. Produce at least one chart whenever the question concerns a trend, a comparison or a
   spatial distribution (a map for spatial questions). Two or three charts at most.
3. If a tool returns an error, read it, fix the arguments and retry (2 times at most); otherwise explain.
4. Before interpreting a monthly drop, check the cloud coverage (coverage_pct or get_quality_report).
5. NTL are a PROXY for activity and electrification: never infer GDP, poverty or electricity access
   directly from them without saying so. Flag small samples and source breaks.
6. Be concise and precise. Round sensibly (2 to 3 significant figures). No table longer than 8 rows.

FINAL ANSWER FORMAT — Markdown, written in {ANSWER_LANGUAGE} (headings translated into that language):
### 🔎 In brief
2 to 3 sentences that answer directly, with the key figures in **bold**.
### 📊 Results
A compact Markdown table or bullet points.
### 🧭 Interpretation
What this means for an official statistician (3 to 5 bullets).
### ⚠️ Limitations and caveats
2 to 4 bullets SPECIFIC to this analysis.
### 🛠️ Method
Tools, products and datasets used (1 to 2 lines).
Do not insert images or links: the charts are displayed automatically below your answer; refer to them by their title."""

### The agent's interface: an answer that reads like an analysis note

An agent's answer intended for a statistics office should read like an **analysis note**, not like a chat thread. Each answer is therefore laid out as follows:

| Block | Role |
|---|---|
| **Header** | Question asked, country, data status (real or demo), model, duration, number of tools |
| **Key indicators** | The 3 to 6 essential figures, computed by the tools (never by the LLM) |
| **Agent's workflow** | The sequence of tools called, in order, with their status: you see *how* the result was obtained |
| **Structured note** | In brief, Results, Interpretation, Limitations and Method, each block with its own style |
| **Numbered figures** | Interactive charts and maps, captioned “Figure n” |
| **Footer** | Sources and a “to be validated before release” notice, plus an expandable audit trail |

In [ ]:
LX_CSS = """<style>
.lx{font-family:Calibri,Carlito,'Segoe UI',Arial,sans-serif;color:#231F20;max-width:1180px;margin:6px 0 4px;
  border-radius:16px;overflow:hidden;border:1px solid #D5DED9;background:#fff;box-shadow:0 4px 18px rgba(0,85,58,.10)}
.lx-head{position:relative;overflow:hidden;padding:20px 26px 24px;color:#fff;
  background:linear-gradient(135deg,#00553A 0%,#00704A 50%,#00A86A 100%)}
.lx-head:after{content:"";position:absolute;left:0;right:0;bottom:0;height:5px;background:#F5C242}
.lx-rings{position:absolute;right:-60px;top:-70px;opacity:.14;pointer-events:none}
.lx-top{display:flex;align-items:center;gap:12px;position:relative}
.lx-avatar{width:44px;height:44px;border-radius:50%;background:#0b2e4f;display:flex;align-items:center;justify-content:center;
  box-shadow:0 0 0 3px rgba(245,194,66,.55),0 0 18px rgba(245,194,66,.55);font-size:22px;flex:none}
.lx-kick{font-size:11px;font-weight:700;letter-spacing:2.5px;text-transform:uppercase;color:#F5C242}
.lx-sub{font-size:12.5px;color:#E6F6EE}
.lx-badge{margin-left:auto;font-size:12px;font-weight:700;padding:5px 12px;border-radius:999px;white-space:nowrap}
.lx-badge.real{background:rgba(255,255,255,.16);border:1px solid rgba(255,255,255,.45);color:#fff}
.lx-badge.demo{background:#B83B2E;border:1px solid #fff;color:#fff}
.lx-q{position:relative;font-size:21px;font-weight:700;line-height:1.3;margin:14px 0 12px;max-width:980px}
.lx-chips{display:flex;flex-wrap:wrap;gap:6px;position:relative}
.lx-chip{font-size:12px;padding:3px 10px;border-radius:999px;background:rgba(255,255,255,.13);border:1px solid rgba(255,255,255,.3)}
.lx-body{padding:18px 26px 8px}
.lx-label{font-size:10.5px;font-weight:700;letter-spacing:2.5px;text-transform:uppercase;color:#00704A;margin:6px 0 8px}
.lx-kpis{display:grid;grid-template-columns:repeat(auto-fit,minmax(170px,1fr));gap:10px;margin-bottom:16px}
.lx-kpi{position:relative;background:#F4F7F5;border:1px solid #D5DED9;border-radius:12px;padding:12px 14px 10px;overflow:hidden}
.lx-kpi:before{content:"";position:absolute;left:0;top:0;bottom:0;width:4px;background:#00A86A}
.lx-kpi:nth-child(2):before{background:#00704A}.lx-kpi:nth-child(3):before{background:#0E7C86}
.lx-kpi:nth-child(4):before{background:#D49A00}.lx-kpi:nth-child(5):before{background:#C4621D}
.lx-kpi .v{font-size:25px;font-weight:700;color:#00553A;line-height:1.1;word-break:break-word}
.lx-kpi .u{font-size:12.5px;color:#5E6964;font-weight:400}
.lx-kpi .l{font-size:12px;color:#5E6964;margin-top:3px}
.lx-kpi .d{font-size:12px;font-weight:700;margin-top:2px}
.lx-up{color:#00A86A}.lx-down{color:#B83B2E}
.lx-flow{display:flex;flex-wrap:wrap;align-items:center;gap:4px;margin-bottom:18px}
.lx-step{display:inline-flex;align-items:center;gap:6px;background:#fff;border:1px solid #D5DED9;border-radius:10px;
  padding:5px 10px;font-size:12.5px}
.lx-step .n{width:18px;height:18px;border-radius:50%;background:#00704A;color:#fff;font-size:10.5px;font-weight:700;
  display:inline-flex;align-items:center;justify-content:center}
.lx-step.err{border-color:#B83B2E;background:rgba(184,59,46,.06)}.lx-step.err .n{background:#B83B2E}
.lx-step .t{color:#8A9590;font-size:11px}
.lx-arrow{color:#A9B5B0;font-size:14px}
.lx-grid{display:grid;grid-template-columns:repeat(auto-fit,minmax(340px,1fr));gap:14px}
.lx-sec{border:1px solid #D5DED9;border-radius:12px;padding:12px 16px;background:#fff;line-height:1.55;font-size:14.5px}
.lx-sec .h{font-size:13px;font-weight:700;letter-spacing:1.5px;text-transform:uppercase;color:#00704A;margin-bottom:6px}
.lx-sec.summary{grid-column:1/-1;background:#E8F5EF;border:1.5px solid #00A86A;font-size:16px}
.lx-sec.summary .h{color:#00553A}
.lx-sec.results{grid-column:1/-1}
.lx-sec.limits{background:rgba(245,194,66,.10);border-color:#D49A00}.lx-sec.limits .h{color:#8a6300}
.lx-sec.method{background:#F4F7F5;color:#5E6964;font-size:13px}.lx-sec.method .h{color:#5E6964}
.lx-sec p{margin:4px 0}.lx-sec ul,.lx-sec ol{margin:4px 0 4px 18px;padding:0}.lx-sec li{margin:2px 0}
.lx-sec table{border-collapse:collapse;width:100%;font-size:13.5px;margin:6px 0}
.lx-sec th{background:#E8F5EF;color:#00553A;text-align:left !important;padding:6px 10px;border-bottom:2px solid #00A86A}
.lx-sec td{padding:5px 10px;border-bottom:1px solid #E1E7E4;text-align:left !important}
.lx-sec tr:nth-child(even) td{background:#FAFCFB}
.lx-sec strong{color:#00553A}
.lx-sec code{background:#F4F7F5;padding:1px 5px;border-radius:4px;font-family:'Courier New',monospace;font-size:12.5px}
.lx-alert{background:rgba(184,59,46,.08);border:1px solid #B83B2E;color:#7a2219;border-radius:10px;padding:9px 14px;margin-bottom:14px}
.lx-foot{display:flex;flex-wrap:wrap;gap:8px 18px;justify-content:space-between;border-top:1px solid #E1E7E4;
  margin-top:16px;padding:10px 0 12px;font-size:11.5px;color:#8A9590}
.lx-figcap{font-family:Calibri,Carlito,'Segoe UI',Arial,sans-serif;display:flex;align-items:center;gap:10px;max-width:1180px;
  margin:14px 0 0;padding:8px 14px;border-radius:10px 10px 0 0;background:#F4F7F5;border:1px solid #D5DED9;border-bottom:3px solid #00A86A}
.lx-figcap .n{font-size:11px;font-weight:700;letter-spacing:2px;text-transform:uppercase;color:#fff;background:#00704A;
  padding:2px 9px;border-radius:999px}
.lx-figcap .t{font-weight:700;color:#231F20;font-size:14px}
.lx-trace{font-family:Calibri,Carlito,'Segoe UI',Arial,sans-serif;max-width:1180px;margin:10px 0 18px;font-size:13px}
.lx-trace summary{cursor:pointer;color:#00704A;font-weight:700}
.lx-trace table{border-collapse:collapse;margin-top:8px;width:100%}
.lx-trace th{background:#E8F5EF;color:#00553A;text-align:left;padding:5px 8px;border-bottom:2px solid #00A86A}
.lx-trace td{padding:4px 8px;border-bottom:1px solid #E1E7E4;text-align:left;vertical-align:top}
.lx-trace code{font-family:'Courier New',monospace;font-size:12px}
.lx-work{font-family:Calibri,Carlito,'Segoe UI',Arial,sans-serif;max-width:1180px;border:1px solid #D5DED9;border-radius:14px;
  overflow:hidden;background:#fff}
.lx-work .bar{height:4px;background:linear-gradient(90deg,#F5C242,#00A86A,#F5C242);background-size:200% 100%;
  animation:lxmove 1.4s linear infinite}
@keyframes lxmove{0%{background-position:0 0}100%{background-position:-200% 0}}
.lx-work .in{padding:12px 18px;display:flex;gap:14px;align-items:flex-start}
.lx-pulse{width:38px;height:38px;border-radius:50%;background:#0b2e4f;display:flex;align-items:center;justify-content:center;
  font-size:19px;flex:none;animation:lxglow 1.6s ease-in-out infinite}
@keyframes lxglow{0%,100%{box-shadow:0 0 0 2px rgba(245,194,66,.4),0 0 6px rgba(245,194,66,.4)}
  50%{box-shadow:0 0 0 4px rgba(245,194,66,.7),0 0 20px rgba(245,194,66,.8)}}
.lx-work .state{font-weight:700;font-size:14.5px;margin-bottom:6px}
</style>"""

LX_RINGS = ('<svg class="lx-rings" width="330" height="330" viewBox="0 0 330 330"><g fill="none" stroke="#fff" '
            'stroke-width="2"><circle cx="165" cy="165" r="160"/><circle cx="165" cy="165" r="118"/>'
            '<circle cx="165" cy="165" r="76"/><circle cx="165" cy="165" r="34"/></g></svg>')

FAMILY_ICON = {"Discovery": "🧭", "Time series": "📈", "Spatial analysis": "🗺️",
               "Population & validation": "👥", "Events & quality": "⚡", "Visualisation": "🎨",
               "Export": "💾", "Extension": "🧩"}
TOOL_LABEL = {
    "get_country_profile": "Country profile", "get_data_availability": "Data availability",
    "list_admin_units": "Administrative units", "get_national_timeseries": "National series",
    "get_admin_timeseries": "Regional series", "detect_anomalies": "Anomaly detection",
    "get_admin_stats": "Regional statistics", "compare_periods": "Period comparison",
    "get_population_light_stats": "Population & light", "get_worldbank_indicator": "World Bank indicator",
    "correlate_ntl_with_indicator": "Proxy validation", "compare_countries": "Country comparison",
    "assess_event_impact": "Event impact", "get_quality_report": "Quality check",
    "plot_timeseries": "Trend chart", "plot_ranking": "Ranking", "plot_map": "Map",
    "plot_scatter": "Scatter plot", "show_night_image": "Night image", "export_dataset": "Export",
    "list_session_datasets": "Session datasets", "get_region_profile": "Region profile",
}
SECTION_KEYS = [("summary", ("bref", "brief", "summary", "resumo", "síntese", "🔎")),
                ("results", ("résultat", "result", "resultado", "📊")),
                ("interp", ("interprét", "interpret", "🧭")),
                ("limits", ("limite", "limit", "précaution", "caveat", "⚠")),
                ("method", ("méthode", "method", "método", "🛠"))]


def _tool_chip(name):
    fam = TOOLS.get(name, {}).get("family", "")
    return f'{FAMILY_ICON.get(fam, "🔧")} {TOOL_LABEL.get(name, name)}'


def _unique_kpis(kpis, n=6):
    seen, out = set(), []
    for k in kpis:
        if k["label"] not in seen:
            seen.add(k["label"])
            out.append(k)
    return out[:n]


def _kpi_html(kpis):
    cells = ""
    for k in kpis:
        d = k.get("delta")
        dh = ""
        if d is not None and not (isinstance(d, float) and math.isnan(d)):
            dh = f'<div class="d {"lx-up" if d >= 0 else "lx-down"}">{"▲" if d >= 0 else "▼"} {abs(d):.1f} %</div>'
        cells += (f'<div class="lx-kpi"><div class="v">{html_lib.escape(str(k["value"]))} '
                  f'<span class="u">{html_lib.escape(str(k.get("unit", "")))}</span></div>'
                  f'<div class="l">{html_lib.escape(k["label"])}</div>{dh}</div>')
    return f'<div class="lx-label">Key indicators</div><div class="lx-kpis">{cells}</div>' if cells else ""


def _flow_html(trace):
    if not trace:
        return ""
    steps = []
    for i, r in enumerate(trace, 1):
        err = "" if r["ok"] else " err"
        mark = "" if r["ok"] else " ⚠"
        steps.append(f'<span class="lx-step{err}" title="{html_lib.escape(r["tool"])}"><span class="n">{i}</span>'
                     f'{_tool_chip(r["tool"])}{mark}<span class="t">{r["duration"]:.1f} s</span></span>')
    return ('<div class="lx-label">Agent\'s workflow</div><div class="lx-flow">'
            + '<span class="lx-arrow">›</span>'.join(steps) + '</div>')


def _sections_html(answer_md):
    """Split the Markdown answer into blocks (In brief, Results, …) and give each one its own style."""
    html = ui.md(answer_md)
    parts = re.split(r"<h[23][^>]*>(.*?)</h[23]>", html, flags=re.S)
    if len(parts) == 1:
        return f'<div class="lx-grid"><div class="lx-sec summary">{html}</div></div>'
    out = f'<div class="lx-sec summary">{parts[0]}</div>' if parts[0].strip() else ""
    for title, body in zip(parts[1::2], parts[2::2]):
        low = title.lower()
        cls = next((c for c, keys in SECTION_KEYS if any(k in low for k in keys)), "interp")
        clean = re.sub(r"^[^\wÀ-ÿ]+", "", re.sub(r"<[^>]+>", "", title)).strip()
        out += f'<div class="lx-sec {cls}"><div class="h">{clean}</div>{body}</div>'
    return f'<div class="lx-grid">{out}</div>'


def _header_html(kicker, sub, question, chips, synthetic):
    badge = ('<span class="lx-badge demo">● DEMO — synthetic data</span>' if synthetic else
             f'<span class="lx-badge real">● Real data · {html_lib.escape(backend.name)}</span>')
    chips_html = "".join(f'<span class="lx-chip">{c}</span>' for c in chips)
    q = f'<div class="lx-q">“{html_lib.escape(question)}”</div>' if question else '<div style="height:10px"></div>'
    return (f'<div class="lx-head">{LX_RINGS}<div class="lx-top"><div class="lx-avatar">🌙</div>'
            f'<div><div class="lx-kick">{kicker}</div><div class="lx-sub">{sub}</div></div>{badge}</div>'
            f'{q}<div class="lx-chips">{chips_html}</div></div>')


def lx_progress_html(state, trace, model, step):
    done = "".join(f'<span class="lx-step{"" if r["ok"] else " err"}"><span class="n">{i}</span>{_tool_chip(r["tool"])}'
                   f'<span class="t">{r["duration"]:.1f} s</span></span>' for i, r in enumerate(trace, 1))
    return (LX_CSS + f'<div class="lx-work"><div class="bar"></div><div class="in"><div class="lx-pulse">🌙</div><div>'
            f'<div class="state">{state}</div><div class="lx-muted" style="font-size:12px;color:#5E6964;margin-bottom:6px">'
            f'Lumina · step {step} · model <code>{html_lib.escape(model)}</code></div>'
            f'<div class="lx-flow" style="margin:0">{done}</div></div></div></div>')


FIG_IN_IFRAME = False
# Inside an output widget (chat, section 10) Colab does not run Plotly's scripts:
# the figure is then placed in a self-contained iframe, which renders everywhere.


def show_figure(fig):
    if not FIG_IN_IFRAME:
        return fig.show()
    h = int(fig.layout.height or 480) + 24
    doc = fig.to_html(include_plotlyjs="cdn", full_html=True, default_width="100%", config={"responsive": True})
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        display(HTML(f'<iframe srcdoc="{html_lib.escape(doc, quote=True)}" '
                     f'style="width:100%;max-width:1180px;height:{h}px;border:0;background:#fff"></iframe>'))


def render_turn(t):
    """Display one agent turn as an analysis note."""
    country = country_name(COUNTRY_ISO3)
    chips = [f"🧠 {html_lib.escape(t['provider'])} · {html_lib.escape(t['model'])}", f"⏱ {t['seconds']:.1f} s",
             f"🛠 {len(t['trace'])} tool(s)", f"🔁 {t['steps']} step(s)"]
    if t["tokens"]:
        chips.append(f"🔤 {fmt(t['tokens'])} tokens")
    if t["figures"]:
        chips.append(f"📊 {len(t['figures'])} figure(s)")
    head = _header_html(f"Lumina · NTL Analyst · {html_lib.escape(country)} ({COUNTRY_ISO3})",
                        f"Question {t['id']} · {t['when']} · STG17 workshop", t["question"], chips, t["synthetic"])
    body = ""
    if t["synthetic"]:
        body += ('<div class="lx-alert"><b>Demo mode:</b> these figures are synthetic and are only meant for '
                 'learning how the agent works. Do not quote them.</div>')
    body += _kpi_html(_unique_kpis(t["kpis"]))
    body += _flow_html(t["trace"])
    body += '<div class="lx-label">Analysis note</div>' + _sections_html(t["answer"])
    datasets = [r["result"] for r in t["trace"] if r["ok"] and r["result"].startswith("ds")]
    foot_left = ("Sources: NOAA/EOG VIIRS, NASA Black Marble, WorldPop, geoBoundaries, World Bank — via "
                 "Google Earth Engine" if not t["synthetic"] else "Source: synthetic data (demo mode)")
    foot_right = "🤖 AI-generated analysis based on traced tools — to be validated by a statistician before release"
    if datasets:
        foot_left += f" · datasets: {', '.join(datasets)}"
    body += f'<div class="lx-foot"><span>{foot_left}</span><span>{foot_right}</span></div>'
    display(HTML(LX_CSS + f'<div class="lx">{head}<div class="lx-body">{body}</div></div>'))

    for n, fid in enumerate(t["figures"], 1):
        f = S.figures[fid]
        display(HTML(LX_CSS + f'<div class="lx-figcap"><span class="n">Figure {n}</span>'
                     f'<span class="t">{html_lib.escape(f["title"])}</span></div>'))
        if f["kind"] == "plotly":
            show_figure(f["obj"])
        else:
            display(HTML(f'<div style="max-width:1180px;background:#000814;border-radius:0 0 12px 12px;padding:14px;'
                         f'text-align:center"><img src="data:image/png;base64,{base64.b64encode(f["obj"]).decode()}" '
                         f'style="max-width:100%;max-height:560px"></div>'))

    if t["trace"]:
        rows = "".join(
            f'<tr><td>{i}</td><td>{_tool_chip(r["tool"])}<br><code>{r["tool"]}</code></td>'
            f'<td><code>{html_lib.escape(json.dumps(r["args"], ensure_ascii=False))[:200]}</code></td>'
            f'<td>{r["duration"]:.1f} s</td><td>{"✅" if r["ok"] else "⚠️"}</td>'
            f'<td style="color:#5E6964">{html_lib.escape(r["result"])}</td></tr>' for i, r in enumerate(t["trace"], 1))
        display(HTML(LX_CSS + f'<details class="lx-trace"><summary>🔍 Audit trail: {len(t["trace"])} tool '
                     f'call(s), exact arguments and results</summary><table><tr><th>#</th><th>Tool</th>'
                     f'<th>Arguments</th><th>Duration</th><th>OK</th><th>Result</th></tr>{rows}</table></details>'))


def render_welcome(a):
    """Agent welcome card: identity, scope and capabilities."""
    fams = {}
    for n in TOOLS:
        if n in a.allowed:
            fams.setdefault(TOOLS[n]["family"], []).append(n)
    chips = [f"🧠 {html_lib.escape(a.llm.provider)} · {html_lib.escape(a.llm.model)}",
             f"🧰 {len(a.allowed)} tools ({a.toolset})", f"🗣 answers in {ANSWER_LANGUAGE}",
             f"📅 annual {min(backend.annual_years)}–{max(backend.annual_years)}"]
    head = _header_html(f"Lumina · NTL Analyst · {html_lib.escape(country_name(a.iso3))} ({a.iso3})",
                        "Agent ready · STG17 workshop · Africa after dark",
                        "Ask me a question about the country's night-time lights: trends, regions, anomalies, "
                        "population, validation…", chips, backend.synthetic)
    cards = "".join(
        f'<div class="lx-sec"><div class="h">{FAMILY_ICON.get(f, "🔧")} {f}</div>'
        f'{", ".join(TOOL_LABEL.get(n, n) for n in ns)}</div>' for f, ns in fams.items())
    display(HTML(LX_CSS + f'<div class="lx">{head}<div class="lx-body"><div class="lx-label">What I can do</div>'
                 f'<div class="lx-grid" style="grid-template-columns:repeat(auto-fit,minmax(230px,1fr))">{cards}</div>'
                 f'<div class="lx-foot"><span>Every figure comes from a traced tool; every answer includes its limitations.'
                 f'</span><span>Try: <code>ask("Which regions have grown the most since 2015?")</code></span>'
                 f'</div></div></div>'))

In [ ]:
class NTLAgent:
    def __init__(self, llm, toolset=TOOLSET, iso3=None, max_steps=MAX_AGENT_STEPS, memory_turns=3):
        self.llm, self.iso3, self.max_steps, self.memory_turns = llm, _iso(iso3), max_steps, memory_turns
        self.set_toolset(toolset)
        self.memory = []

    def set_toolset(self, toolset):
        self.toolset = toolset
        self.allowed = {n for n, t in TOOLS.items() if toolset == "full" or t["core"]}
        self.schemas = [TOOLS[n]["schema"] for n in TOOLS if n in self.allowed]

    def reset(self):
        self.memory = []
        UI.ok("New conversation: the agent's memory has been cleared.")

    def _assistant_message(self, msg, step):
        calls = []
        for i, c in enumerate(msg.tool_calls):
            tc = {"id": c.id or f"call_{step}_{i}", "type": "function",
                  "function": {"name": c.function.name, "arguments": c.function.arguments or "{}"}}
            extra = (getattr(c, "model_extra", None) or {}).get("extra_content")
            if extra and self.llm.provider == "gemini":        # Gemini thought signature, to be sent back unchanged
                tc["extra_content"] = extra
            calls.append(tc)
        return {"role": "assistant", "content": msg.content or None, "tool_calls": calls}

    def ask(self, question, show=True):
        S.new_turn()
        fig_before = set(S.figures)
        messages = [{"role": "system", "content": build_system_prompt(self.iso3)}]
        for q, a in self.memory[-self.memory_turns:]:
            messages += [{"role": "user", "content": q}, {"role": "assistant", "content": a}]
        messages.append({"role": "user", "content": question})
        trace, tokens, t0, answer, fixes = [], 0, time.time(), "", 0
        live = display(HTML(""), display_id=True) if show else None

        def progress(state):
            if live is not None:
                live.update(HTML(lx_progress_html(state, trace, self.llm.model, step)))

        step = 0
        while step < self.max_steps:
            step += 1
            progress("Thinking about the best approach…" if step == 1 else "Analysing the results obtained…")
            try:
                resp = self.llm.chat(messages, tools=self.schemas)
            except openai.BadRequestError as e:
                if fixes < 2 and ("tool" in str(e).lower() or "function" in str(e).lower()):
                    fixes += 1
                    messages.append({"role": "user", "content": "Your last tool call was malformed. "
                                     "Try again, following the JSON schema of the parameters exactly."})
                    continue
                raise
            if getattr(resp, "usage", None):
                tokens += (resp.usage.prompt_tokens or 0) + (resp.usage.completion_tokens or 0)
            msg = resp.choices[0].message
            if not msg.tool_calls:
                answer = (msg.content or "").strip()
                if answer:
                    break
                messages.append({"role": "user", "content": "Now write the final answer in the required format."})
                continue
            messages.append(self._assistant_message(msg, step))
            for tc in messages[-1]["tool_calls"]:
                name, raw = tc["function"]["name"], tc["function"]["arguments"]
                try:
                    args = json.loads(raw) if raw and raw.strip() else {}
                except json.JSONDecodeError:
                    args = None
                progress(f"Running: {_tool_chip(name)}…")
                result, rec = execute_tool(name, args, self.allowed)
                rec["step"] = step
                trace.append(rec)
                messages.append({"role": "tool", "tool_call_id": tc["id"], "content": result})
        if not answer:
            messages.append({"role": "user", "content": "Maximum number of steps reached: write the final answer "
                             "with the results already obtained, in the required format."})
            answer = (self.llm.chat(messages).choices[0].message.content or "").strip() or "_(no answer)_"

        turn = dict(id=S.turn_id, question=question, answer=answer, trace=trace, kpis=list(S.kpis),
                    figures=[f for f in S.figures if f not in fig_before], provider=self.llm.provider,
                    model=self.llm.model, seconds=time.time() - t0, steps=step, tokens=tokens,
                    toolset=self.toolset, synthetic=backend.synthetic, when=dt.datetime.now().strftime("%Y-%m-%d %H:%M"))
        S.turns.append(turn)
        self.memory.append((question, answer))
        if live is not None:
            live.update(HTML(""))
        if show:
            render_turn(turn)
        return turn


agent = NTLAgent(llm) if llm else None


def ask(question, **kw):
    """Shortcut: ask the session agent a question."""
    if agent is None:
        return ui.warn("No LLM connected: go back to section 04.")
    agent.ask(question, **kw)          # the full turn remains available in S.turns[-1]


if agent:
    render_welcome(agent)
else:
    ui.warn("Agent not created: no LLM connected (section 04).")

In [ ]:
# 👀 Curious? Here is the exact system prompt sent to the LLM.
ui.card("System prompt", f"<pre>{html_lib.escape(build_system_prompt())}</pre>", kicker="Under the hood")

## 09 · Guided demonstrations

Each cell asks a question of increasing difficulty. Watch the **trace** to see which tools the agent chooses, in what order, and whether it corrects itself after an error. Feel free to modify the questions.

> ⏱️ With Earth Engine, the first computation for a country takes from a few seconds to a few minutes depending on its size; results are then **cached** (`ntl_agent_outputs/cache`).

In [ ]:
# 1️⃣ National trend
ask("How has national night-time brightness evolved over the last ten available years? "
    "Give the trend, the compound annual growth rate (CAGR) and a chart.")

In [ ]:
# 2️⃣ Geography of light
ask("Which regions (ADM1) are the most and least lit in the latest available year? "
    "Show a map of mean radiance and a ranking, and say whether light is highly concentrated.")

In [ ]:
# 3️⃣ Subnational dynamics
ask("Which regions progressed the most between 2015 and the latest available year? "
    "Are the least-lit regions catching up with the others? Support this with a map of changes.")

In [ ]:
# 4️⃣ Monthly anomalies — real drops or clouds?
ask("Were there any abnormal months in the last four years? Distinguish likely real drops "
    "from cloud-related artefacts, with a chart.")

In [ ]:
# 5️⃣ Proxy validation (Day 4, afternoon)
ask("Are night-time lights a good proxy for GDP in this country? Validate against World Bank data "
    "and conclude: usable for dissemination or only as a diagnostic tool?")

In [ ]:
# 6️⃣ Population in the dark and comparison with peer countries
PEERS = "KEN,UGA,TZA,BDI"   # ← adapt to your country's neighbours
ask(f"What share of the population lives in unlit areas, by region? Then compare {COUNTRY_ISO3} "
    f"with {PEERS} on SOL per capita and access to electricity.")

In [ ]:
# 7️⃣ Impact of a dated event (daily Black Marble) — enter a date that is meaningful for your country
EVENT_DATE = "2023-05-03"   # ← e.g. flood, national blackout, cyclone, conflict
ask(f"Assess the impact on night-time lighting of the event of {EVENT_DATE} (30-day windows before/after), "
    "by region, with a map.")

## 10 · Chat with the agent

A conversation interface: type your question or click on a suggestion. The agent remembers the last three exchanges, so you can follow up (“and for the districts?”, “make a map of it”).
If the widgets do not display in your environment, simply use `ask("your question")`.

In [ ]:
import ipywidgets as W

SUGGESTIONS = [
    "Show me the country at night for the latest available year",
    "Rank the districts (ADM2) by lit area",
    "Produce a data quality report for last year",
    "Compare 2016 and the latest year at district level",
    "What is the correlation with access to electricity?",
    "Export the last table to Excel",
]


def chat_ui():
    if agent is None:
        return ui.warn("No LLM connected (section 04).")
    box = W.Textarea(placeholder="E.g.: Which regions have progressed the most since 2015? Show a map.",
                     layout=W.Layout(width="100%", height="80px", margin="10px 0 6px"))
    send = W.Button(description="Ask Lumina", icon="paper-plane", layout=W.Layout(width="210px", height="38px"))
    if "text_color" in send.style.keys:          # ipywidgets ≥ 8: brand colours
        send.style.button_color, send.style.text_color, send.style.font_weight = "#00704A", "white", "bold"
    else:                                          # ipywidgets 7 (older Colab versions)
        send.button_style = "success"
    new = W.Button(description="New conversation", icon="refresh", layout=W.Layout(width="210px", height="38px"))
    out = W.Output()
    chips = []
    for s in SUGGESTIONS:
        b = W.Button(description="✦ " + s[:46] + ("…" if len(s) > 46 else ""), tooltip=s,
                     layout=W.Layout(width="auto", margin="3px"))
        b.style.button_color = "#E8F5EF"
        b.on_click(lambda _b, s=s: setattr(box, "value", s))
        chips.append(b)

    def on_send(_):
        q = box.value.strip()
        if not q:
            return
        global FIG_IN_IFRAME
        send.disabled = True
        FIG_IN_IFRAME = True
        with out:
            try:
                agent.ask(q)
            except Exception as e:
                ui.alert(f"Error: <code>{html_lib.escape(str(e)[:400])}</code>")
        FIG_IN_IFRAME = False
        send.disabled = False
        box.value = ""

    def on_new(_):
        with out:
            out.clear_output()
            agent.reset()

    send.on_click(on_send)
    new.on_click(on_new)
    header = W.HTML(LX_CSS + '<div class="lx" style="margin-bottom:0">' + _header_html(
        f"Lumina · Conversation · {html_lib.escape(country_name(COUNTRY_ISO3))} ({COUNTRY_ISO3})",
        "Type your question or pick a suggestion — the agent remembers the last 3 exchanges",
        "", [f"🧠 {html_lib.escape(agent.llm.model)}", f"🧰 {len(agent.allowed)} tools",
             f"🗣 {ANSWER_LANGUAGE}"], backend.synthetic) + '</div>')
    display(W.VBox([header, W.HBox(chips, layout=W.Layout(flex_flow="row wrap")), box, W.HBox([send, new]), out]))


chat_ui()

## 11 · Lab: comparing providers

Same question, same toolbox, different providers: we measure **latency**, **number of steps**, **tokens consumed** and **success**. Only providers whose key is available (or Ollama if it is running) are tested — no key is requested here.
Record your results in the workshop's shared comparison sheet (Day 2 · *Choosing your engine*).

In [ ]:
BENCH_QUESTION = ("What is the change in the national sum of radiances between 2016 and the latest available year, "
                  "and the compound annual growth rate (CAGR)?")


def provider_available(p):
    if p == "ollama":
        try:
            return requests.get(PROVIDERS["ollama"]["base_url"].replace("/v1", "/api/tags"), timeout=2).ok
        except Exception:
            return False
    return bool(get_secret(PROVIDERS[p]["key_env"], prompt=False))


def run_benchmark(question=BENCH_QUESTION, providers=("groq", "gemini", "ollama"), toolset="core"):
    rows = []
    for p in providers:
        if not provider_available(p):
            rows.append(dict(provider=p, model="—", status="not configured"))
            continue
        try:
            a = NTLAgent(LLMClient(p, prompt_for_key=False), toolset=toolset)
            t = a.ask(question, show=False)
            rows.append(dict(provider=p, model=t["model"], status="✅", latency_s=round(t["seconds"], 1),
                             steps=t["steps"], tool_calls=len(t["trace"]), tokens=t["tokens"],
                             tool_errors=sum(not r["ok"] for r in t["trace"]),
                             answer=t["answer"][:160].replace("\n", " ") + "…"))
        except Exception as e:
            rows.append(dict(provider=p, model=PROVIDERS[p]["default_model"], status=f"❌ {str(e)[:80]}"))
    df = pd.DataFrame(rows)
    ui.card("Provider comparison", df.to_html(index=False, na_rep="—", escape=True), kicker="11 · Benchmark")
    ok = df[df.status == "✅"] if "latency_s" in df else pd.DataFrame()
    if len(ok):
        fig = go.Figure(go.Bar(x=ok.provider, y=ok.latency_s, marker_color=SERIES[:len(ok)],
                               text=[f"{v:.1f} s" for v in ok.latency_s], textposition="outside"))
        _style(fig, "Agent response time by provider", "seconds", None, height=380).show()
    return df

bench = run_benchmark()

## 12 · Mini evaluation bench

“It looks right” is not enough. We build a small **evaluation set**: for each question, the **ground truth is computed by Python** with the same tools, then we check that the agent's answer (a) called a relevant tool and (b) contains the correct value. This is the approach presented on Day 2 (*Prompt optimisation*): measure before optimising.

In [ ]:
def _truth_top_region():
    S.new_turn()
    return get_admin_stats()["concentration"]["top1_unit"]


def _truth_n_adm1():
    return str(len(backend.admin_units(COUNTRY_ISO3, 1)))


def _truth_last_year():
    return str(backend.default_year())


EVAL_SET = [
    dict(question="How many level-1 administrative units does the country have? Answer briefly.",
         tools={"get_country_profile", "list_admin_units"}, truth=_truth_n_adm1),
    dict(question="Which region was the brightest (SOL) in the latest available year? Answer briefly.",
         tools={"get_admin_stats"}, truth=_truth_top_region),
    dict(question="What is the latest year of annual data available? Answer briefly.",
         tools={"get_country_profile", "get_data_availability", "get_national_timeseries"}, truth=_truth_last_year),
]


def run_eval(a=None):
    a = a or agent
    if a is None:
        return ui.warn("No LLM connected.")
    rows = []
    for case in EVAL_SET:
        truth = case["truth"]()
        t = a.ask(case["question"], show=False)
        a.memory.clear()                      # independent questions
        used = {r["tool"] for r in t["trace"]}
        norm = lambda s: re.sub(r"\s+", " ", s.lower())
        rows.append(dict(question=case["question"][:70] + "…", truth=truth,
                         relevant_tool="✅" if used & case["tools"] else "❌",
                         correct_value="✅" if norm(truth) in norm(t["answer"]) else "❌",
                         tools=", ".join(sorted(used)) or "none", latency_s=round(t["seconds"], 1)))
    df = pd.DataFrame(rows)
    score = (df.relevant_tool.eq("✅").mean() + df.correct_value.eq("✅").mean()) / 2
    ui.card(f"Score: {score:.0%}", df.to_html(index=False, escape=True), kicker="12 · Evaluation",
            highlight=score >= 0.8)
    return df

eval_df = run_eval()

## 13 · Publish: a self-contained HTML report

All the session's questions, answers, interactive charts and traces are assembled into **a single HTML file**: share it by email or publish it directly on **GitHub Pages** (Day 5). The report states the sources and the limitations.

In [ ]:
def generate_report(path=None, title=None):
    path = Path(path or OUT / f"ntl_report_{COUNTRY_ISO3}_{dt.date.today():%Y%m%d}.html")
    title = title or f"Night-time lights — {country_name(COUNTRY_ISO3)}"
    first_plotly = True
    sections = ""
    for t in S.turns:
        figs = ""
        for fid in t["figures"]:
            f = S.figures[fid]
            if f["kind"] == "plotly":
                figs += f["obj"].to_html(full_html=False, include_plotlyjs="cdn" if first_plotly else False)
                first_plotly = False
            else:
                figs += (f'<div style="background:#000814;border-radius:10px;padding:12px;text-align:center;margin:10px 0">'
                         f'<img src="data:image/png;base64,{base64.b64encode(f["obj"]).decode()}" style="max-width:100%"></div>')
        trace = "".join(f'<li><code>{r["tool"]}</code> {html_lib.escape(json.dumps(r["args"], ensure_ascii=False))[:160]}'
                        f' — {r["duration"]:.1f} s {"✅" if r["ok"] else "⚠️"}</li>' for r in t["trace"])
        sections += (f'<div class="lx" style="margin:18px 0 0">'
                     + _header_html(f"Lumina · Question {t['id']}", f"{t['when']} · {t['provider']} · {t['model']}",
                                    t["question"], [f"⏱ {t['seconds']:.1f} s", f"🛠 {len(t['trace'])} tool(s)"],
                                    t["synthetic"])
                     + f'<div class="lx-body">{_kpi_html(_unique_kpis(t["kpis"]))}{_flow_html(t["trace"])}'
                     f'<div class="lx-label">Analysis note</div>{_sections_html(t["answer"])}'
                     f'<div class="lx-label" style="margin-top:14px">Figures</div>{figs}'
                     f'<details class="lx-trace"><summary>Tool trace</summary><ul>{trace}</ul></details>'
                     f'</div></div>')
    if not sections:
        sections = '<div class="ntl-warn">No questions asked yet.</div>'
    warn = ('<div class="ntl-alert"><b>Synthetic data (demo mode)</b> — demonstration report, '
            'figures are not real.</div>') if backend.synthetic else ""
    doc = f"""<!doctype html><html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1"><title>{html_lib.escape(title)}</title>{BASE_CSS}{LX_CSS}
<style>body{{margin:0;background:#F4F7F5}} .wrap{{max-width:1050px;margin:0 auto;padding:0 18px 40px}}</style></head>
<body class="ntl"><div style="background:linear-gradient(135deg,#00553A,#00704A 45%,#00A86A);color:#fff;
padding:44px 20px 50px;position:relative"><div class="wrap" style="padding-bottom:0">
<div style="letter-spacing:3px;font-size:12px;font-weight:700;color:#F5C242">STG17 · NTL ANALYSIS REPORT</div>
<div style="font-size:38px;font-weight:700;margin:8px 0">{html_lib.escape(title)}</div>
<div style="font-style:italic;color:#E6F6EE">Produced by an AI agent ({llm.provider if llm else '—'} ·
{llm.model if llm else '—'}) · data: {backend.name} · {dt.date.today():%d/%m/%Y}</div></div>
<div style="position:absolute;left:0;right:0;bottom:0;height:7px;background:#F5C242"></div></div>
<div class="wrap">{warn}{sections}
<div class="ntl-card"><div class="ntl-kicker">Method, sources and limitations</div>
<p>VIIRS radiances (NOAA/EOG VNL v2 annual, VCMSLCFG monthly), NASA Black Marble VNP46A2, WorldPop Global2 100 m (2015–2030),
geoBoundaries v6, World Bank WDI — processed with Google Earth Engine. “Lit” threshold: {LIT_THRESHOLD} nW/cm²/sr.</p>
<ul><li>Night-time lights are a <b>proxy</b>: they do not directly measure GDP, poverty or
access to electricity.</li><li>Possible artefacts: clouds, seasonality, gas flares, urban halo, rural noise,
product breaks.</li><li>Each answer was generated by an LLM from the tool results traced above;
it must be <b>checked by a statistician</b> before any official dissemination.</li></ul></div></div></body></html>"""
    path.write_text(doc, encoding="utf-8")
    ui.ok(f"Report saved: <code>{path}</code> ({path.stat().st_size / 1024:.0f} KB, {len(S.turns)} question(s)).")
    if IN_COLAB:
        try:
            from google.colab import files
            files.download(str(path))
        except Exception:
            pass
    return path

report_path = generate_report()

## 14 · Extending the toolbox

Adding a skill to the agent = writing **one well-documented Python function** and decorating it with `@tool`. Example: a **region fact sheet** (rank, share of national SOL, 5-year change). Then recreate the agent so that it sees the new tool.

**Exercise ideas**: urban area expansion (pixels > 10 nW), detection of newly lit areas between two years, pixel-level intra-regional inequality index, comparison with census results, historical DMSP-OLS series (1992–2013) with a sensor-break warning.

In [ ]:
@tool("Extension")
def get_region_profile(region: str, level: int = 1, iso3: Optional[str] = None) -> dict:
    """Fact sheet for ONE region: national rank, share of national SOL, mean radiance, lit share and
    change over the last 5 years. The region name is automatically matched to the official name.

    Args:
        region: Region name (tolerates typos and missing accents).
        level: Administrative level: 1 = regions, 2 = districts.
        iso3: ISO3 country code. Default: the session country.
    """
    iso, level = _iso(iso3), _level(level)
    last = backend.default_year()
    cur = backend.zonal_stats(iso, level, [last - 5, last])
    names = _match_units([region], cur.unit.unique())
    if not names:
        raise ValueError(f"Region “{region}” not found. Examples: {list(cur.unit.unique()[:6])}")
    now = cur[cur.year == last].copy()
    now["sol_share"] = 100 * now.sol / now.sol.sum()
    now["rank"] = now.sol.rank(ascending=False).astype(int)
    r = now[now.unit == names[0]].iloc[0]
    past = cur[(cur.year == last - 5) & (cur.unit == names[0])].sol.iloc[0]
    S.add_kpi(f"Rank of {names[0]}", f"{r['rank']}/{len(now)}")
    S.add_kpi("Share of national SOL", f"{r.sol_share:.1f}", "%")
    return {"region": names[0], "year": last, "rank": int(r["rank"]), "of": len(now), "sol": r.sol,
            "sol_share_pct": r.sol_share, "mean_rad": r.mean_rad, "lit_share_pct": r.lit_share,
            "sol_change_5y_pct": 100 * (r.sol / past - 1) if past else None, "notes": _base_notes()}


if llm:
    agent = NTLAgent(llm, toolset="full")       # the agent now sees the get_region_profile tool
    example_region = backend.admin_units(COUNTRY_ISO3, 1).unit.iloc[0]
    ask(f"Give me the fact sheet for the region {example_region}.")

## 15 · Limitations, ethics and good practice

| Issue | What this notebook does | What remains your responsibility |
|---|---|---|
| **Hallucinations** | No figure without a tool; full trace; automatic evaluation | Review every answer before dissemination; check key figures in the exported datasets |
| **Proxy validity** | Correlation tool, systematic caution notes | Validate at **subnational** level against your official statistics and document the “dissemination or diagnostic” decision |
| **Confidentiality** | Only public aggregates are sent to the LLM | Never send microdata to a cloud LLM; prefer Ollama for sensitive data |
| **Licences** | Sources cited in every chart and in the report | Respect the licences: VIIRS/EOG (CC BY 4.0), WorldPop (CC BY 4.0), geoBoundaries (per-country licences), World Bank (CC BY 4.0) |
| **Reproducibility** | Centralised parameters, cache, exportable datasets, model and date in the report | Version the notebook and outputs on GitHub; record the LLM model version (models change) |
| **Costs and quotas** | Disk cache, `core` toolset, retry on 429 | Monitor Earth Engine and LLM quotas; estimate the cost per analysis before scaling up |

### Further reading

* Elvidge C.D. *et al.* (2021), *Annual time series of global VIIRS nighttime lights derived from monthly averages: 2012 to 2019*, **Remote Sensing** 13(5) — VNL v2 product.
* Román M.O. *et al.* (2018), *NASA's Black Marble nighttime lights product suite*, **Remote Sensing of Environment** 210.
* Runfola D. *et al.* (2020), *geoBoundaries: A global database of political administrative boundaries*, **PLoS ONE** 15(4).
* Earth Engine catalogue: <https://developers.google.com/earth-engine/datasets/catalog/NOAA_VIIRS_DNB_ANNUAL_V22> · <https://developers.google.com/earth-engine/datasets/catalog/NOAA_VIIRS_DNB_MONTHLY_V1_VCMSLCFG> · <https://developers.google.com/earth-engine/datasets/catalog/NASA_VIIRS_002_VNP46A2>
* World Bank, *Light Every Night* and the *Open Nighttime Lights* guide: <https://worldbank.github.io/OpenNightLights/>
* OpenAI compatibility: Groq <https://console.groq.com/docs/openai> · Gemini <https://ai.google.dev/gemini-api/docs/openai> · Ollama <https://github.com/ollama/ollama/blob/main/docs/openai.md>

---
*Educational notebook produced for the STG17 technical workshop “Emerging Issues, Emerging Practice”. Visual identity inspired by the AfDB, unofficial.*